# **Battery State of Health Estimation Using Deep Learning Modles with Multi-Parameter Feature Engineering**

**Project Overview:**

This notebook implements various machine learning algorithms for battery State of Health (SoH) estimation using the NASA Battery Aging Dataset. The research focuses on developing robust estimation algorithms through advanced feature engineering and comparative analysis of multiple DL approaches.

**Data Source:**

NASA Battery Dataset from Kaggle: [Dataset Link](https://www.kaggle.com/datasets/patrickfleith/nasa-battery-dataset)

---

### **1. Package Loading and Environment Setup**

#### **1.1 Import Core Libraries**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Data manipulation and analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from tqdm.notebook import tqdm 
import tensorflow as tf
import random

# File handling and utilities
import os
import glob
import ipywidgets
from IPython.display import display, clear_output
from pathlib import Path

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
tf.config.experimental.enable_op_determinism()

# Add GPU memory growth configuration
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)


#### **1.2 GPU Configuration**

In [ ]:
from pynvml import *

# Function to get GPU memory usage
def get_gpu_memory_usage():
    try:
        nvmlInit()
        handle = nvmlDeviceGetHandleByIndex(0)
        info = nvmlDeviceGetMemoryInfo(handle)
        used_memory_gb = info.used / 1024**3
        total_memory_gb = info.total / 1024**3
        return used_memory_gb, total_memory_gb
    except NVMLError as error:
        print(f"Failed to query GPU memory: {error}")
        return None, None
    finally:
        try:
            nvmlShutdown()
        except NVMLError:
            pass

# Now, call the function
used, total = get_gpu_memory_usage()
if used is not None:
    print(f"GPU Memory Used: {used:.2f} GB / {total:.2f} GB")

### **2. Data Loading and Initial Inspectiion**

#### **2.1 Dataset Path Configuration**

In [ ]:
# Define data paths
BASE_PATH = "" # Used for Google Colab Mount
METADATA_PATH = os.path.join(BASE_PATH, "cleaned_dataset/metadata.csv")
DATA_FOLDER = os.path.join(BASE_PATH, "cleaned_dataset/data")
EXPORT_DIR = os.path.join(BASE_PATH, 'export')
PLOT_DIR = os.path.join(EXPORT_DIR, 'plots')

os.makedirs(EXPORT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# Verify paths exist
if not os.path.exists(METADATA_PATH):
    raise FileNotFoundError(f"Metadata file not found at: {METADATA_PATH}")
if not os.path.exists(DATA_FOLDER):
    raise FileNotFoundError(f"Data folder not found at: {DATA_FOLDER}")

print(f"Base path: {BASE_PATH}")
print(f"Metadata path: {METADATA_PATH}")
print(f"Data folder: {DATA_FOLDER}")
print(f"All Processed data will be saved in: {EXPORT_DIR}/")

#### **2.2 Metadata Loading and Inspection**

In [ ]:
# Load metadata
metadata = pd.read_csv(METADATA_PATH)

print("Metadata Dataset Information (Before Type Conversion):")
print("=" * 50)
print(f"Dataset shape: {metadata.shape}")
print(f"Memory usage: {metadata.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nColumn information:")
print(metadata.dtypes)
print("\n" + "=" * 50)

# Display first few rows of metadata
print("First few rows of metadata:")
display(metadata.head())

#### **2.3 Data Type Conversion and Metadata Preprocessing**

In [ ]:
# Convert numerical columns from object to appropriate types
def preprocess_metadata(df):
    """
    Preprocess metadata by converting data types and handling missing values
    """
    df = df.copy()

    # Convert Capacity, Re, Rct to numeric (they are stored as strings)
    numeric_columns = ['Capacity', 'Re', 'Rct']
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Parse the weird start_time format
    def parse_start_time(time_str):
        """Convert '[2010. 7. 21. 15. 0. 35.093]' to datetime"""
        try:
            # Remove brackets and split by whitespace
            parts = time_str.strip('[]').split()
            # Convert to integers (except seconds which may have decimals)
            year = int(float(parts[0]))
            month = int(float(parts[1]))
            day = int(float(parts[2]))
            hour = int(float(parts[3]))
            minute = int(float(parts[4]))
            second = float(parts[5])
            
            # Create datetime
            dt = pd.Timestamp(year=year, month=month, day=day, 
                            hour=hour, minute=minute, second=int(second))
            # Add microseconds if any
            microseconds = int((second - int(second)) * 1e6)
            dt = dt + pd.Timedelta(microseconds=microseconds)
            return dt
        except:
            return pd.NaT
    
    # Apply to start_time column
    df['start_time'] = df['start_time'].apply(parse_start_time)

    return df

# Apply preprocessing
metadata = preprocess_metadata(metadata)

print("Metadata Dataset Information (After Type Conversion):")
print("=" * 50)
print(f"Dataset shape: {metadata.shape}")
print(f"Memory usage: {metadata.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nColumn information:")
print(metadata.dtypes)
print("\nMissing values:")
print(metadata.isnull().sum())
print("First few rows of metadata:")
display(metadata.head())

#### **2.4 Battery Identification and Data Overview**

In [ ]:
# Identify unique batteries in the dataset
unique_batteries = metadata['battery_id'].unique()
print(f"Available batteries: {sorted(unique_batteries)}")
print(f"Total number of batteries: {len(unique_batteries)}")

# Analyze test types distribution
test_type_counts = metadata['type'].value_counts()
print(f"\nTest type distribution:")
print(test_type_counts)

# Battery-wise data distribution
battery_distribution = metadata.groupby('battery_id').agg({
    'type': 'count',
    'Capacity': lambda x: x.notna().sum(),
    'Re': lambda x: x.notna().sum(),
    'Rct': lambda x: x.notna().sum()
}).rename(columns={'type': 'total_cycles'})

print(f"\nBattery-wise data distribution:")
display(battery_distribution)

#### **2.5 Sample Cycle Data Loading and Inspection**

In [ ]:
# Load a sample cycle file to understand the structure
sample_file = os.path.join(DATA_FOLDER, "00001.csv")

if os.path.exists(sample_file):
    sample_cycle = pd.read_csv(sample_file)

    print("Sample Cycle Data Information:")
    print("=" * 40)
    print(f"Dataset shape: {sample_cycle.shape}")
    print(f"Memory usage: {sample_cycle.memory_usage(deep=True).sum() / 1024:.2f} KB")
    print("\nColumn information:")
    print(sample_cycle.dtypes)

    print(f"\nFirst few rows of sample cycle data:")
    display(sample_cycle.head())

    print(f"\nBasic statistics:")
    display(sample_cycle.describe())
else:
    print(f"Sample file {sample_file} not found")

#### **2.6 Analyzing Impedance Data**

In [ ]:
# Analyzing impedance data structure
impedance_df = metadata[metadata['type'] == 'impedance'][['battery_id', 'test_id', 'start_time', 'Re', 'Rct']].dropna(subset=['Re', 'Rct'])

print(f"Impedance measurements with valid Re/Rct: {len(impedance_df)}")
print(f"battery_id unique values: {impedance_df['battery_id'].nunique()}, that is = {impedance_df['battery_id'].unique()}")
print(f"Sample impedance data:")
print(impedance_df.head())

# Check if the issue is NaN values in original impedance rows
print(f"\nChecking impedance rows:")
impedance_all = metadata[metadata['type'] == 'impedance']
print(f"Total impedance rows: {len(impedance_all)}")
print(f"Impedance with Re values: {impedance_all['Re'].notna().sum()}")
print(f"Impedance with Rct values: {impedance_all['Rct'].notna().sum()}")

#### **2.7 Merging Impedance Data with Cycle Data**

In [ ]:
def map_impedance_to_discharge_vectorized(metadata_df, max_time_diff_hours=12):
    """
    Vectorized mapping of Re and Rct from impedance measurements to discharge cycles
    Optimized for fast processing on limited VRAM systems
    
    Args:
        metadata_df: DataFrame containing both discharge and impedance data
        max_time_diff_hours: Maximum time difference (in hours) to consider for mapping
        
    Returns:
        DataFrame with mapped Re and Rct values for discharge cycles
    """
    # Separate discharge and impedance data
    discharge_df = metadata_df[metadata_df['type'] == 'discharge'].copy()
    impedance_df = metadata_df[
        (metadata_df['type'] == 'impedance') & 
        (metadata_df['Re'].notna()) & 
        (metadata_df['Rct'].notna())
    ].copy()
    
    # Initialize result columns
    discharge_df['Re_mapped'] = np.nan
    discharge_df['Rct_mapped'] = np.nan
    discharge_df['impedance_time_diff'] = np.nan
    
    print(f"Processing {len(discharge_df)} discharge cycles with vectorized operations...")
    
    # Process each battery separately to optimize memory usage
    for battery_id in tqdm(discharge_df['battery_id'].unique()):
        # Filter data for current battery
        battery_discharge = discharge_df[discharge_df['battery_id'] == battery_id].copy()
        battery_impedance = impedance_df[impedance_df['battery_id'] == battery_id].copy()
        
        if len(battery_impedance) == 0:
            continue
            
        # Vectorized time difference calculation using broadcasting
        discharge_times = battery_discharge['start_time'].values[:, np.newaxis]
        impedance_times = battery_impedance['start_time'].values[np.newaxis, :]
        
        # Calculate all pairwise time differences in hours
        time_diffs = (impedance_times - discharge_times) / np.timedelta64(1, 'h')
        
        # Find valid measurements within time window
        valid_mask = np.abs(time_diffs) <= max_time_diff_hours
        
        # Process each discharge cycle for this battery
        for i, discharge_idx in enumerate(battery_discharge.index):
            valid_impedance_indices = np.where(valid_mask[i])[0]
            
            if len(valid_impedance_indices) == 0:
                continue
                
            # Get corresponding impedance measurements
            valid_impedance = battery_impedance.iloc[valid_impedance_indices]
            
            # Calculate averages
            discharge_df.loc[discharge_idx, 'Re_mapped'] = valid_impedance['Re'].mean()
            discharge_df.loc[discharge_idx, 'Rct_mapped'] = valid_impedance['Rct'].mean()
            
            # Store minimum time difference
            min_time_diff = np.abs(time_diffs[i, valid_impedance_indices]).min()
            discharge_df.loc[discharge_idx, 'impedance_time_diff'] = min_time_diff
    
    # Apply reasonable bounds filtering
    reasonable_bounds = {
        'Re_mapped': (0.01, 1.0),
        'Rct_mapped': (0.01, 2.0)
    }

    for col, (min_val, max_val) in reasonable_bounds.items():
        mask = (discharge_df[col] < min_val) | (discharge_df[col] > max_val)
        filtered_count = mask.sum()
        if filtered_count > 0:
            print(f"Filtering {filtered_count} unrealistic {col} values")
        discharge_df.loc[mask, col] = np.nan
    
    # Calculate mapping statistics
    mapped_cycles = discharge_df['Re_mapped'].notna().sum()
    total_cycles = len(discharge_df)
    mapping_ratio = mapped_cycles / total_cycles * 100
    
    print("\nMapping Statistics:")
    print(f"Total discharge cycles: {total_cycles}")
    print(f"Cycles with mapped impedance: {mapped_cycles}")
    print(f"Mapping success rate: {mapping_ratio:.1f}%")
    
    # Analyze time differences
    time_diffs = discharge_df['impedance_time_diff'].dropna()
    if len(time_diffs) > 0:
        print(f"\nTime difference statistics (hours):")
        print(f"Mean: {time_diffs.mean():.2f}")
        print(f"Median: {time_diffs.median():.2f}")
        print(f"Min: {time_diffs.min():.2f}")
        print(f"Max: {time_diffs.max():.2f}")
    
    return discharge_df

# Map impedance data to discharge cycles
discharge_with_impedance = map_impedance_to_discharge_vectorized(metadata)

#### **2.8 Validate Impedance Mapping**

In [ ]:
def validate_impedance_mapping(discharge_with_impedance, metadata):
    """
    Validate that impedance mapping is correct by comparing with original data
    """
    print("Impedance Mapping Validation")
    print("=" * 60)
    
    # Check specific battery B0047 early cycles for manual verification
    validation_battery = 'B0047'
    validation_cycles = discharge_with_impedance[
        (discharge_with_impedance['battery_id'] == validation_battery) & 
        (discharge_with_impedance['test_id'] <= 10)
    ].copy()
    
    print(f"Validation for battery {validation_battery} (first 10 cycles):")
    print(f"{'Test_ID':<8} {'Type':<10} {'Start_Time':<20} {'Re_mapped':<12} {'Rct_mapped':<12} {'Time_Diff':<10}")
    print("-" * 80)
    
    for _, row in validation_cycles.iterrows():
        print(f"{row['test_id']:<8} {'discharge':<10} {str(row['start_time'])[:19]:<20} "
              f"{row['Re_mapped']:<12.6f} {row['Rct_mapped']:<12.6f} {row['impedance_time_diff']:<10.2f}")
    
    # Show corresponding impedance measurements
    impedance_ref = metadata[
        (metadata['battery_id'] == validation_battery) & 
        (metadata['type'] == 'impedance') & 
        (metadata['test_id'] <= 10) &
        (metadata['Re'].notna())
    ]
    
    print(f"\nCorresponding impedance measurements:")
    print(f"{'Test_ID':<8} {'Type':<10} {'Start_Time':<20} {'Re_original':<12} {'Rct_original':<12}")
    print("-" * 80)
    
    for _, row in impedance_ref.iterrows():
        print(f"{row['test_id']:<8} {'impedance':<10} {str(row['start_time'])[:19]:<20} "
              f"{row['Re']:<12.6f} {row['Rct']:<12.6f}")
    
    # Verify mapping accuracy for first few cycles
    print(f"\nMapping Accuracy Check:")
    print("-" * 40)
    
    # Check discharge cycle 0 should map to impedance cycle 1
    discharge_0 = validation_cycles[validation_cycles['test_id'] == 0].iloc[0]
    impedance_1 = impedance_ref[impedance_ref['test_id'] == 1].iloc[0]
    
    print(f"Discharge cycle 0 mapped Re: {discharge_0['Re_mapped']:.6f}")
    print(f"Impedance cycle 1 actual Re: {impedance_1['Re']:.6f}")
    print(f"Difference: {abs(discharge_0['Re_mapped'] - impedance_1['Re']):.6f}")
    
    print(f"\nDischarge cycle 0 mapped Rct: {discharge_0['Rct_mapped']:.6f}")
    print(f"Impedance cycle 1 actual Rct: {impedance_1['Rct']:.6f}")
    print(f"Difference: {abs(discharge_0['Rct_mapped'] - impedance_1['Rct']):.6f}")
    
    return validation_cycles, impedance_ref

# Run validation
validation_cycles, impedance_ref = validate_impedance_mapping(discharge_with_impedance, metadata)

#### **2.9 Impedance Anomalies Detection**

In [ ]:
def analyze_impedance_anomalies(discharge_with_impedance):
    """
    Analyze impedance values for anomalies and data quality issues
    """
    print("Impedance Anomaly Analysis")
    print("=" * 60)
    
    # Get cycles with valid impedance mappings
    valid_impedance = discharge_with_impedance[
        discharge_with_impedance['Re_mapped'].notna() & 
        discharge_with_impedance['Rct_mapped'].notna()
    ].copy()
    
    print(f"Total cycles with impedance data: {len(valid_impedance)}")
    
    # Statistical analysis
    re_stats = valid_impedance['Re_mapped'].describe()
    rct_stats = valid_impedance['Rct_mapped'].describe()
    
    print(f"\nRe Statistics (Ω):")
    print(re_stats)
    print(f"\nRct Statistics (Ω):")
    print(rct_stats)
    
    # Detect outliers using IQR method
    def detect_outliers(data, column_name):
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = data[(data < lower_bound) | (data > upper_bound)]
        return outliers, lower_bound, upper_bound
    
    re_outliers, re_lower, re_upper = detect_outliers(valid_impedance['Re_mapped'], 'Re_mapped')
    rct_outliers, rct_lower, rct_upper = detect_outliers(valid_impedance['Rct_mapped'], 'Rct_mapped')
    
    print(f"\nRe outliers: {len(re_outliers)} cycles")
    print(f"Re normal range: {re_lower:.6f} - {re_upper:.6f} Ω")
    
    print(f"\nRct outliers: {len(rct_outliers)} cycles")
    print(f"Rct normal range: {rct_lower:.6f} - {rct_upper:.6f} Ω")
    
    # Visualizations
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Re analysis
    axes[0,0].hist(valid_impedance['Re_mapped'], bins=50, alpha=0.7, color='blue')
    axes[0,0].axvline(re_lower, color='red', linestyle='--', label=f'Lower bound: {re_lower:.6f}')
    axes[0,0].axvline(re_upper, color='red', linestyle='--', label=f'Upper bound: {re_upper:.6f}')
    axes[0,0].set_title('Re Distribution')
    axes[0,0].set_xlabel('Re (Ω)')
    axes[0,0].legend()
    
    # Rct analysis
    axes[0,1].hist(valid_impedance['Rct_mapped'], bins=50, alpha=0.7, color='green')
    axes[0,1].axvline(rct_lower, color='red', linestyle='--', label=f'Lower bound: {rct_lower:.6f}')
    axes[0,1].axvline(rct_upper, color='red', linestyle='--', label=f'Upper bound: {rct_upper:.6f}')
    axes[0,1].set_title('Rct Distribution')
    axes[0,1].set_xlabel('Rct (Ω)')
    axes[0,1].legend()
    
    # Time difference analysis
    axes[0,2].hist(valid_impedance['impedance_time_diff'], bins=30, alpha=0.7, color='orange')
    axes[0,2].set_title('Impedance Mapping Time Difference')
    axes[0,2].set_xlabel('Time Difference (hours)')
    
  # Battery-wise impedance trends
    for battery in valid_impedance['battery_id'].unique()[:5]:  # Show first 5 batteries
        battery_data = valid_impedance[valid_impedance['battery_id'] == battery]
        if len(battery_data) > 5:  # Only if sufficient data
            axes[1,0].plot(battery_data['test_id'], battery_data['Re_mapped'], 
                        alpha=0.7, label=battery)
    axes[1,0].set_title('Re Evolution by Battery')
    axes[1,0].set_xlabel('Test ID')
    axes[1,0].set_ylabel('Re (Ω)')
    axes[1,0].legend()

    for battery in valid_impedance['battery_id'].unique()[:5]:
        battery_data = valid_impedance[valid_impedance['battery_id'] == battery]
        if len(battery_data) > 5:
            axes[1,1].plot(battery_data['test_id'], battery_data['Rct_mapped'], 
                        alpha=0.7, label=battery)
    axes[1,1].set_title('Rct Evolution by Battery')
    axes[1,1].set_xlabel('Test ID')
    axes[1,1].set_ylabel('Rct (Ω)')
    axes[1,1].legend()
    
    # Re vs Rct correlation
    axes[1,2].scatter(valid_impedance['Re_mapped'], valid_impedance['Rct_mapped'], alpha=0.5)
    axes[1,2].set_title('Re vs Rct Relationship')
    axes[1,2].set_xlabel('Re (Ω)')
    axes[1,2].set_ylabel('Rct (Ω)')
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, 'impedance_anomaly_analysis.png'))
    plt.show()
    
    return valid_impedance, re_outliers, rct_outliers

# Run impedance anomaly analysis
valid_impedance_data, re_outliers, rct_outliers = analyze_impedance_anomalies(discharge_with_impedance)

### **3. Cycle Data Processing and Capacity Calculation**

#### **3.1 Import Additional Libraries for Cycle Processing**

In [ ]:
# Additional imports for this section
from scipy import integrate
from scipy.signal import find_peaks
import concurrent.futures
from typing import Dict, List, Tuple, Optional
from scipy.stats import pearsonr

#### **3.2 Cycle Data Loading Functions**

In [ ]:
def load_cycle_data(filename: str) -> pd.DataFrame:
    """
    Load individual cycle data from CSV file
    """
    filepath = os.path.join(DATA_FOLDER, filename)
    if os.path.exists(filepath):
        return pd.read_csv(filepath)
    else:
        return None

def calculate_capacity_coulomb_counting(cycle_df: pd.DataFrame) -> float:
    """
    Calculate capacity using Coulomb counting method
    Capacity = ∫|Current| dt (in Ah)
    """
    if cycle_df is None or len(cycle_df) < 2:
        return np.nan

    # Sort by time to ensure proper integration
    cycle_df = cycle_df.sort_values('Time').reset_index(drop=True)

    # Calculate capacity using trapezoidal integration
    current_discharge = cycle_df['Current_measured'].values  
    time_hours = cycle_df['Time'].values / 3600  # Convert seconds to hours

    # Integrate using trapezoidal rule
    capacity_ah = np.trapz(-current_discharge, time_hours)

    return capacity_ah

def extract_cycle_features(cycle_df: pd.DataFrame) -> Dict[str, float]:
    """
    Extract key features from a single cycle
    """
    if cycle_df is None or len(cycle_df) == 0:
        return {}

    features = {}

    # Basic statistics
    features['voltage_start'] = cycle_df['Voltage_measured'].iloc[0]
    features['voltage_end'] = cycle_df['Voltage_measured'].iloc[-1]
    features['voltage_mean'] = cycle_df['Voltage_measured'].mean()
    features['voltage_min'] = cycle_df['Voltage_measured'].min()
    features['voltage_drop'] = features['voltage_start'] - features['voltage_end']

    # Current features
    features['current_mean'] = cycle_df['Current_measured'].mean()
    features['current_std'] = cycle_df['Current_measured'].std()

    # Temperature features
    features['temperature_mean'] = cycle_df['Temperature_measured'].mean()
    features['temperature_max'] = cycle_df['Temperature_measured'].max()
    features['temperature_range'] = (cycle_df['Temperature_measured'].max() -
                                     cycle_df['Temperature_measured'].min())

    # Time features
    features['duration_seconds'] = cycle_df['Time'].max()
    features['duration_hours'] = features['duration_seconds'] / 3600

    # Energy delivered (Wh)
    if len(cycle_df) > 1:
        power = np.abs(cycle_df['Voltage_measured'] * cycle_df['Current_measured'])
        time_hours = cycle_df['Time'] / 3600
        features['energy_wh'] = np.trapz(power, time_hours)
    else:
        features['energy_wh'] = 0

    return features

#### **3.3 Process Sample Cycles and Validate Capacity Calculation**

In [ ]:

# Test on a few discharge cycles with known capacity
test_cycles = metadata[
    (metadata['type'] == 'discharge') &
    (metadata['Capacity'].notna())
].head(10)

print("Validating capacity calculation on sample discharge cycles:")
print("=" * 70)

validation_results = []

for idx, row in test_cycles.iterrows():
    cycle_data = load_cycle_data(row['filename'])

    if cycle_data is not None:
        # Calculate capacity
        calculated_capacity = calculate_capacity_coulomb_counting(cycle_data)
        metadata_capacity = row['Capacity']

        # Calculate error
        error = abs(calculated_capacity - metadata_capacity)
        percent_error = (error / metadata_capacity) * 100 if metadata_capacity > 0 else np.nan

        validation_results.append({
            'filename': row['filename'],
            'battery_id': row['battery_id'],
            'metadata_capacity': metadata_capacity,
            'calculated_capacity': calculated_capacity,
            'absolute_error': error,
            'percent_error': percent_error
        })

validation_df = pd.DataFrame(validation_results)

def validate_capacity_calculation(validation_df: pd.DataFrame):
    """Add IEEE-style validation metrics"""
    r2 = np.corrcoef(validation_df['metadata_capacity'], validation_df['calculated_capacity'])[0,1]**2
    rmse = np.sqrt(np.mean((validation_df['metadata_capacity'] - validation_df['calculated_capacity'])**2))
    mape = np.mean(np.abs((validation_df['metadata_capacity'] - validation_df['calculated_capacity']) / validation_df['metadata_capacity'])) * 100
    
    print(f"R²: {r2:.4f}")
    print(f"RMSE: {rmse:.4f} Ah") 
    print(f"MAPE: {mape:.2f}%")
    

print(f"\nValidation Sample Results:")
print(validation_df.to_string(index=False))

print(f"\nMetric summary:")
print("=" * 70)
validate_capacity_calculation(validation_df)

r_value, p_value = pearsonr(validation_df['metadata_capacity'], validation_df['calculated_capacity'])
print(f"\nCorrelation coefficient: {r_value:.4f} (p-value: {p_value:.2e})")

#### **3.4 Extract Features for Sample Cycles**

In [ ]:
# Extract features for the same sample cycles
print("\nExtracting features from sample cycles:")
print("=" * 50)

sample_features = []

for idx, row in test_cycles.head(3).iterrows():
    cycle_data = load_cycle_data(row['filename'])

    if cycle_data is not None:
        features = extract_cycle_features(cycle_data)
        features['filename'] = row['filename']
        features['battery_id'] = row['battery_id']
        features['calculated_capacity'] = calculate_capacity_coulomb_counting(cycle_data)
        features['dq_dv_max'] = np.max(np.gradient(cycle_data['Voltage_measured'], cycle_data['Time']))
        features['voltage_variance'] = cycle_data['Voltage_measured'].var()
        features['current_efficiency'] = len(cycle_data[cycle_data['Current_measured'] < -0.1]) / len(cycle_data)
        sample_features.append(features)

# Display sample features
features_df = pd.DataFrame(sample_features)
print("\nSample extracted features:")
for col in features_df.columns:
    if col not in ['filename', 'battery_id']:
        print(f"{col}: {features_df[col].values}")

#### **3.5 Analyze Battery Cycle**

In [ ]:
def analyze_battery_quality(metadata_df: pd.DataFrame, min_cycles: int = 25) -> pd.DataFrame:
    """
    Analyze battery quality metrics for informed exclusion decisions.
    Flags batteries with total cycles <= min_cycles or missing capacity data.
    """
    # Get all discharge cycles
    discharge_cycles = metadata_df[metadata_df['type'] == 'discharge'].copy()
    
    battery_analysis = []
    
    for battery_id in discharge_cycles['battery_id'].unique():
        battery_data = discharge_cycles[discharge_cycles['battery_id'] == battery_id]
        
        # Count cycles with capacity data
        n_cycles_total = len(battery_data)
        n_cycles_with_capacity = battery_data['Capacity'].notna().sum()
        
        # Capacity statistics
        capacity_mean = battery_data['Capacity'].mean()
        capacity_std = battery_data['Capacity'].std()
        capacity_cv = capacity_std / capacity_mean if capacity_mean > 0 else np.inf
        
        # Early cycle variation
        early_cycles = battery_data.iloc[:10]
        early_capacity_std = early_cycles['Capacity'].std() if len(early_cycles) > 1 else np.nan
        
        battery_analysis.append({
            'battery_id': battery_id,
            'n_cycles_total': n_cycles_total,
            'n_cycles_with_capacity': n_cycles_with_capacity,
            'capacity_mean': capacity_mean,
            'capacity_std': capacity_std,
            'capacity_cv': capacity_cv,
            'early_capacity_std': early_capacity_std,
            'has_insufficient_data': n_cycles_total <= min_cycles,  # <= instead of <
            'has_no_capacity_data': n_cycles_with_capacity == 0
        })
    
    analysis_df = pd.DataFrame(battery_analysis)
    return analysis_df.sort_values('n_cycles_total')

battery_analysis = analyze_battery_quality(metadata, min_cycles=25)

print("\nBattery Analysis Summary:")
print("=" * 50)

# Identify problematic batteries (<= min_cycles or no capacity)
print("Problematic batteries ( ≤ 25 cycles or no capacity data):")
problematic = battery_analysis[
    (battery_analysis['has_insufficient_data']) | 
    (battery_analysis['has_no_capacity_data'])
]
print(problematic[['battery_id', 'n_cycles_total', 'n_cycles_with_capacity']].to_string(index=False))

# Suggested batteries to exclude
suggested_exclusions = problematic['battery_id'].tolist()
print(f"\nSuggested batteries to exclude: {suggested_exclusions}")

#### **3.6 Process All Discharge Cycles**

In [ ]:
def calculate_capacity_with_voltage_threshold(cycle_df: pd.DataFrame,
                                             voltage_threshold: float = 2.7) -> float:
    """
    Calculate capacity using Coulomb counting with voltage threshold
    Stops integration when voltage drops below threshold
    """
    if cycle_df is None or len(cycle_df) < 2:
        return np.nan

    # Sort by time
    cycle_df = cycle_df.sort_values('Time').reset_index(drop=True)

    # Find where voltage drops below threshold
    below_threshold = cycle_df['Voltage_measured'] < voltage_threshold

    if below_threshold.any():
        # Get index of first occurrence below threshold
        cutoff_idx = below_threshold.idxmax()
        # Use data only up to threshold
        cycle_df = cycle_df.iloc[:cutoff_idx + 1]

    if len(cycle_df) < 2:
        return np.nan

    # Calculate capacity
    current_discharge = cycle_df['Current_measured'].values
    time_hours = cycle_df['Time'].values / 3600

    capacity_ah = np.trapz(-current_discharge, time_hours)

    return capacity_ah

def process_discharge_cycles_refined(metadata_df: pd.DataFrame,
                                   exclude_batteries: List[str] = suggested_exclusions,
                                   voltage_threshold: float = 2.7,
                                   capacity_bounds: Tuple[float, float] = (0.5, 2.5)) -> pd.DataFrame:
    """
    Process discharge cycles with refined filtering and validation
    """
    # Filter discharge cycles and exclude problematic batteries
    discharge_cycles = metadata_df[
        (metadata_df['type'] == 'discharge') &
        (~metadata_df['battery_id'].isin(exclude_batteries))
    ].copy()

    print(f"Total discharge cycles to process (excluding {exclude_batteries}): {len(discharge_cycles)}")
    print("=" * 50)

    all_features = []
    anomaly_count = 0

    for idx, row in tqdm(discharge_cycles.iterrows(), total=len(discharge_cycles), desc="Processing cycles"):
        cycle_data = load_cycle_data(row['filename'])

        if cycle_data is not None and len(cycle_data) > 10:
            # Calculate capacity with voltage threshold
            calculated_capacity = calculate_capacity_with_voltage_threshold(cycle_data, voltage_threshold)

            # Check if capacity is within reasonable bounds
            if capacity_bounds[0] <= calculated_capacity <= capacity_bounds[1]:
                # Extract features
                features = extract_cycle_features(cycle_data)
                features['calculated_capacity'] = calculated_capacity

                # Add metadata
                features['filename'] = row['filename']
                features['battery_id'] = row['battery_id']
                features['test_id'] = row['test_id']
                features['uid'] = row['uid']
                features['ambient_temperature'] = row['ambient_temperature']
                features['metadata_capacity'] = row['Capacity']
                # Add impedance features from mapped data
                impedance_row = discharge_with_impedance[discharge_with_impedance['filename'] == row['filename']]
                if not impedance_row.empty:
                    features['Re_mapped'] = impedance_row['Re_mapped'].iloc[0]
                    features['Rct_mapped'] = impedance_row['Rct_mapped'].iloc[0]
                    features['impedance_time_diff'] = impedance_row['impedance_time_diff'].iloc[0]
                else:
                    features['Re_mapped'] = np.nan
                    features['Rct_mapped'] = np.nan
                    features['impedance_time_diff'] = np.nan

                # Additional features for voltage profile
                features['time_to_2_7v'] = (cycle_data[cycle_data['Voltage_measured'] < voltage_threshold]['Time'].min()
                                           if any(cycle_data['Voltage_measured'] < voltage_threshold) else cycle_data['Time'].max())

                all_features.append(features)
            else:
                anomaly_count += 1

    features_df = pd.DataFrame(all_features)

    # Add cycle number for each battery
    features_df['cycle_number'] = features_df.groupby('battery_id').cumcount() + 1

    print(f"\nProcessed {len(features_df)} discharge cycles successfully")
    print(f"Excluded {anomaly_count} cycles with anomalous capacity values")

    return features_df

# Reprocess with refined method
discharge_features_refined = process_discharge_cycles_refined(metadata)

#### **3.7 Compare Calculated Capacity with Metadata**

In [ ]:
def plot_capacity_distribution(features_df: pd.DataFrame):
    """
    Plot the distribution of calculated capacities
    """
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=features_df, x='metadata_capacity', y='calculated_capacity', alpha=0.6)
    plt.title('Capacity Comparison: Metadata vs Calculated')
    plt.xlabel('Metadata Capacity (Ah)')
    plt.ylabel('Calculated Capacity (Ah)')
    plt.plot([0, features_df['metadata_capacity'].max()], [0, features_df['metadata_capacity'].max()], 
             color='red', linestyle='--', label='y=x')
    plt.legend()
    plt.savefig(os.path.join(PLOT_DIR, 'capacity_comparison.png'))
    plt.show() 

def validate_capacity_calculation_refined(validation_df_refined: pd.DataFrame):
    """Add IEEE-style validation metrics for refined method"""
    r2 = np.corrcoef(validation_df_refined['metadata_capacity'], validation_df_refined['calculated_capacity'])[0,1]**2
    rmse = np.sqrt(np.mean((validation_df_refined['metadata_capacity'] - validation_df_refined['calculated_capacity'])**2))
    mape = np.mean(np.abs((validation_df_refined['metadata_capacity'] - validation_df_refined['calculated_capacity']) / validation_df_refined['metadata_capacity'])) * 100
    
    print(f"R²: {r2:.4f}")
    print(f"RMSE: {rmse:.4f} Ah") 
    print(f"MAPE: {mape:.4f}%")

print("\nValidation Sample Results (Refined Method):")
validation_df_refined = discharge_features_refined[['filename', 'battery_id', 'metadata_capacity', 'calculated_capacity']]
print(validation_df_refined.to_string(index=False))
print("\nMetric summary (Refined Method):")
validate_capacity_calculation_refined(validation_df_refined)

r_value_refined, p_value_refined = pearsonr(validation_df_refined['metadata_capacity'], validation_df_refined['calculated_capacity'])
print(f"\nCorrelation coefficient (Refined Method): {r_value_refined:.4f} (p-value: {p_value_refined:.2e})")

plot_capacity_distribution(discharge_features_refined)

#### **3.7 Analyze Processed Data**

In [ ]:
print("\nRefined Processing Results:")
print("=" * 50)
print(f"Total valid discharge cycles: {len(discharge_features_refined)}")
print(f"Number of batteries: {discharge_features_refined['battery_id'].nunique()}")
print(f"\nCapacity statistics (refined):")
print(f"Mean capacity: {discharge_features_refined['calculated_capacity'].mean():.3f} Ah")
print(f"Std capacity: {discharge_features_refined['calculated_capacity'].std():.3f} Ah")
print(f"Min capacity: {discharge_features_refined['calculated_capacity'].min():.3f} Ah")
print(f"Max capacity: {discharge_features_refined['calculated_capacity'].max():.3f} Ah")

# Check excluded batteries are not present
excluded_batteries = suggested_exclusions
batteries_in_data = discharge_features_refined['battery_id'].unique()
print(f"\nExcluded batteries check:")
for bat in excluded_batteries:
    print(f"{bat}: {'NOT in data ✓' if bat not in batteries_in_data else 'Still in data ✗'}")

#### **3.8 Anomaly Detection and Visualization**

In [ ]:
def detect_capacity_anomalies(battery_data: pd.DataFrame,
                             max_increase: float = 0.02,
                             max_decrease: float = 0.15,
                             rolling_window: int = 5) -> pd.DataFrame:
    """
    Detect anomalies in capacity degradation
    """
    battery_data = battery_data.sort_values('cycle_number').copy()

    # Calculate capacity change
    battery_data['capacity_change'] = battery_data['calculated_capacity'].diff()
    battery_data['capacity_change_pct'] = battery_data['calculated_capacity'].pct_change()

    # Calculate rolling statistics
    battery_data['capacity_rolling_mean'] = battery_data['calculated_capacity'].rolling(
        window=rolling_window, center=True, min_periods=1).mean()
    battery_data['capacity_rolling_std'] = battery_data['calculated_capacity'].rolling(
        window=rolling_window, center=True, min_periods=1).std()

    # Flag anomalies
    battery_data['is_spike'] = battery_data['capacity_change_pct'] > max_increase
    battery_data['is_drop'] = battery_data['capacity_change_pct'] < -max_decrease
    battery_data['is_outlier'] = (
        np.abs(battery_data['calculated_capacity'] - battery_data['capacity_rolling_mean']) >
        2 * battery_data['capacity_rolling_std']
    )

    battery_data['is_anomaly'] = (
        battery_data['is_spike'] |
        battery_data['is_drop'] |
        battery_data['is_outlier']
    )

    return battery_data

# Analyze anomalies for all batteries
anomaly_summary = []

for battery_id in discharge_features_refined['battery_id'].unique():
    battery_data = discharge_features_refined[
        discharge_features_refined['battery_id'] == battery_id
    ].copy()

    battery_data = detect_capacity_anomalies(battery_data)

    anomaly_summary.append({
        'battery_id': battery_id,
        'total_cycles': len(battery_data),
        'anomaly_count': battery_data['is_anomaly'].sum(),
        'anomaly_percentage': (battery_data['is_anomaly'].sum() / len(battery_data)) * 100,
        'spike_count': battery_data['is_spike'].sum(),
        'drop_count': battery_data['is_drop'].sum(),
        'outlier_count': battery_data['is_outlier'].sum()
    })

anomaly_df = pd.DataFrame(anomaly_summary).sort_values('anomaly_percentage', ascending=False)

def classify_battery_health(anomaly_df):
    """Classify batteries by health quality for IEEE reporting"""
    conditions = [
        (anomaly_df['anomaly_percentage'] <= 5),
        (anomaly_df['anomaly_percentage'] <= 10),
        (anomaly_df['anomaly_percentage'] <= 20),
        (anomaly_df['anomaly_percentage'] > 20)
    ]
    choices = ['Excellent', 'Good', 'Fair', 'Poor']
    anomaly_df['health_grade'] = np.select(conditions, choices)
    return anomaly_df

classify_battery_health(anomaly_df)

print("Battery Anomaly Analysis:")
print("=" * 80)
display(anomaly_df.head(10))

#### **3.9 Visualize Anomalies in Detail**

In [ ]:
# Visualize anomalies for problematic batteries
problematic_batteries = anomaly_df[anomaly_df['anomaly_percentage'] > 5]['battery_id'].head(4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, battery_id in enumerate(problematic_batteries):
    battery_data = discharge_features_refined[
        discharge_features_refined['battery_id'] == battery_id
    ].copy()
    battery_data = detect_capacity_anomalies(battery_data)

    ax = axes[idx]

    # Plot all points
    ax.scatter(battery_data['cycle_number'],
              battery_data['calculated_capacity'],
              alpha=0.6, s=30, label='All data')

    # Highlight anomalies
    anomalies = battery_data[battery_data['is_anomaly']]
    ax.scatter(anomalies['cycle_number'],
              anomalies['calculated_capacity'],
              color='red', s=50, label='Anomalies', zorder=5)

    # Plot rolling mean
    ax.plot(battery_data['cycle_number'],
           battery_data['capacity_rolling_mean'],
           'g--', alpha=0.7, label='Rolling mean')
    
    # Add confidence intervals to rolling mean
    ax.fill_between(battery_data['cycle_number'],
                    battery_data['capacity_rolling_mean'] - 2*battery_data['capacity_rolling_std'],
                    battery_data['capacity_rolling_mean'] + 2*battery_data['capacity_rolling_std'],
                    alpha=0.2, color='green', label='95% CI')

    ax.set_xlabel('Cycle Number')
    ax.set_ylabel('Capacity (Ah)')
    ax.set_title(f'Battery {battery_id} - Anomaly Detection')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'anomaly_detection.png'))
plt.show()

#### **3.10 Apply Anomaly Filtering**

In [ ]:
def filter_anomalous_cycles(features_df: pd.DataFrame,
                          max_increase: float = 0.02,
                          max_decrease: float = 0.15) -> pd.DataFrame:
    """
    Filter out anomalous cycles from the dataset
    """
    filtered_data = []

    for battery_id in features_df['battery_id'].unique():
        battery_data = features_df[features_df['battery_id'] == battery_id].copy()
        battery_data = battery_data.sort_values('cycle_number')

        # Detect anomalies
        battery_data = detect_capacity_anomalies(battery_data, max_increase, max_decrease)

        # Keep only non-anomalous data
        clean_data = battery_data[~battery_data['is_anomaly']]

        # Recalculate cycle numbers after filtering
        clean_data = clean_data.copy()
        clean_data['cycle_number'] = range(1, len(clean_data) + 1)

        filtered_data.append(clean_data)

    return pd.concat(filtered_data, ignore_index=True)

# Apply filtering
discharge_features_clean = filter_anomalous_cycles(discharge_features_refined)

print(f"Original cycles: {len(discharge_features_refined)}")
print(f"Clean cycles: {len(discharge_features_clean)}")
print(f"Removed: {len(discharge_features_refined) - len(discharge_features_clean)} anomalous cycles")

### **4. Voltage Profile Extraction and Interpolation**

#### **4.1 Import Additional Libraries for Deep Learning Preparation**

In [ ]:
from scipy.interpolate import interp1d
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pickle

#### **4.2 Raw Discharge Profile**

In [ ]:
def get_raw_discharge_profile(cycle_df: pd.DataFrame,
                              voltage_threshold: float = 2.7) -> Dict[str, np.ndarray]:
    """
    Extract raw discharge profile (Voltage, Current, Temperature) before interpolation.

    Parameters:
    - cycle_df: Single cycle DataFrame (from load_cycle_data)
    - voltage_threshold: Cutoff voltage to trim the discharge curve (default=2.7 V)

    Returns:
    - Dict containing raw time (seconds), voltage, current, temperature arrays
    """
    if cycle_df is None or len(cycle_df) < 10:
        return None

    # Sort by time to ensure correct order
    cycle_df = cycle_df.sort_values('Time').reset_index(drop=True)

    # Apply voltage threshold to cut the discharge tail
    below_threshold = cycle_df['Voltage_measured'] < voltage_threshold
    if below_threshold.any():
        cutoff_idx = below_threshold.idxmax()
        cycle_df = cycle_df.iloc[:cutoff_idx + 1]

    if len(cycle_df) < 10:
        return None

    raw_data = {
        'time': cycle_df['Time'].values,
        'voltage': cycle_df['Voltage_measured'].values,
        'current': cycle_df['Current_measured'].values,
        'temperature': cycle_df['Temperature_measured'].values
    }
    return raw_data


In [ ]:
# Example: visualize raw data for one cycle
sample_row = discharge_features_clean.iloc[0]
cycle_data = load_cycle_data(sample_row['filename'])

raw_profile = get_raw_discharge_profile(cycle_data)

if raw_profile:
    plt.figure(figsize=(15, 4))

    # Voltage
    plt.subplot(1, 3, 1)
    plt.plot(raw_profile['time'], raw_profile['voltage'])
    plt.xlabel('Time (s)')
    plt.ylabel('Voltage (V)')
    plt.title('Raw Voltage Profile')
    plt.grid(True, alpha=0.3)

    # Current
    plt.subplot(1, 3, 2)
    plt.plot(raw_profile['time'], raw_profile['current'])
    plt.xlabel('Time (s)')
    plt.ylabel('Current (A)')
    plt.title('Raw Current Profile')
    plt.grid(True, alpha=0.3)

    # Temperature
    plt.subplot(1, 3, 3)
    plt.plot(raw_profile['time'], raw_profile['temperature'])
    plt.xlabel('Time (s)')
    plt.ylabel('Temperature (°C)')
    plt.title('Raw Temperature Profile')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, 'raw_discharge_profile.png'))
    plt.show()


#### **4.3 Extract Full Voltage Profiles with Interpolation**

In [ ]:
def interpolate_discharge_profile(cycle_df: pd.DataFrame,
                                 n_points: int = 100,
                                 voltage_threshold: float = 2.7) -> Dict[str, np.ndarray]:
    """
    Interpolate discharge profile to fixed number of points using normalized time
    """
    if cycle_df is None or len(cycle_df) < 10:
        return None

    # Sort by time
    cycle_df = cycle_df.sort_values('Time').reset_index(drop=True)

    # Apply voltage threshold
    below_threshold = cycle_df['Voltage_measured'] < voltage_threshold
    if below_threshold.any():
        cutoff_idx = below_threshold.idxmax()
        cycle_df = cycle_df.iloc[:cutoff_idx + 1]

    if len(cycle_df) < 10:
        return None

    # Normalize time to [0, 1]
    time_normalized = (cycle_df['Time'] - cycle_df['Time'].min()) / (cycle_df['Time'].max() - cycle_df['Time'].min())

    # Create interpolation points
    time_interp = np.linspace(0, 1, n_points)

    # Interpolate each variable
    interpolated_data = {}

    # Voltage interpolation
    f_voltage = interp1d(time_normalized, cycle_df['Voltage_measured'], kind='linear', fill_value='extrapolate')
    interpolated_data['voltage'] = f_voltage(time_interp)

    # Current interpolation
    f_current = interp1d(time_normalized, cycle_df['Current_measured'], kind='linear', fill_value='extrapolate')
    interpolated_data['current'] = f_current(time_interp)

    # Temperature interpolation
    f_temp = interp1d(time_normalized, cycle_df['Temperature_measured'], kind='linear', fill_value='extrapolate')
    interpolated_data['temperature'] = f_temp(time_interp)

    # Time in seconds (actual discharge duration)
    interpolated_data['time_seconds'] = cycle_df['Time'].max()

    return interpolated_data

# Test interpolation on a sample cycle
sample_row = discharge_features_clean.iloc[0]
cycle_data = load_cycle_data(sample_row['filename'])
interpolated = interpolate_discharge_profile(cycle_data)

if interpolated:
    print("Sample interpolated profile:")
    print(f"Voltage shape: {interpolated['voltage'].shape}")
    print(f"Current shape: {interpolated['current'].shape}")
    print(f"Temperature shape: {interpolated['temperature'].shape}")
    print(f"Discharge duration: {interpolated['time_seconds']:.2f} seconds")

    # Visualize interpolation
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.plot(interpolated['voltage'])
    plt.xlabel('Normalized Time Points')
    plt.ylabel('Voltage (V)')
    plt.title('Interpolated Voltage Profile')
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 3, 2)
    plt.plot(interpolated['current'])
    plt.xlabel('Normalized Time Points')
    plt.ylabel('Current (A)')
    plt.title('Interpolated Current Profile')
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 3, 3)
    plt.plot(interpolated['temperature'])
    plt.xlabel('Normalized Time Points')
    plt.ylabel('Temperature (°C)')
    plt.title('Interpolated Temperature Profile')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, 'interpolated_profile.png'))
    plt.show()

#### **4.4 Feature Corrrelation Analysis**

In [ ]:
print("Feature Correlation Analysis")
print("="*60)

# Select numerical features for correlation analysis
feature_cols = [
    'calculated_capacity',  # Target variable
    'ambient_temperature',
    'duration_hours',
    'energy_wh',
    'voltage_drop',
    'temperature_mean',
    'temperature_range',
    'voltage_start',
    'voltage_end',
    'voltage_mean',
    'current_mean',
    'cycle_number',
    'Re_mapped',
    'Rct_mapped',
    'impedance_time_diff'
]

# Create correlation matrix
correlation_data = discharge_features_clean[feature_cols].copy()
correlation_matrix = correlation_data.corr()

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# 1. Full correlation heatmap
sns.heatmap(correlation_matrix, 
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm', 
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8},
            ax=ax1)
ax1.set_title('Feature Correlation Matrix', fontsize=10, pad=20)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')

# 2. Correlation with capacity only
capacity_corr = correlation_matrix['calculated_capacity'].sort_values(ascending=False)
colors = ['red' if abs(corr) > 0.9 else 'orange' if abs(corr) > 0.7 else 'green' 
          for corr in capacity_corr.values]

bars = ax2.barh(capacity_corr.index, capacity_corr.values, color=colors)
ax2.set_xlabel('Correlation with Capacity')
ax2.set_title('Feature Correlations with Battery Capacity', fontsize=14)
ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlim(-1, 1)

# Add value labels on bars
for bar, value in zip(bars, capacity_corr.values):
    x_pos = value + (0.02 if value > 0 else -0.02)
    ax2.text(x_pos, bar.get_y() + bar.get_height()/2, 
             f'{value:.3f}', 
             ha='left' if value > 0 else 'right',
             va='center',
             fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'feature_correlation_analysis.png'))
plt.show()

# Print warning about high correlations
print("\nCORRELATION WARNINGS:")
high_corr_features = []
for feature in feature_cols[1:]:  # Skip capacity itself
    corr_value = correlation_matrix.loc['calculated_capacity', feature]
    if abs(corr_value) > 0.9:
        print(f"   - {feature}: {corr_value:.3f} (VERY HIGH - Potential leakage!)")
        high_corr_features.append(feature)
    elif abs(corr_value) > 0.7:
        print(f"   - {feature}: {corr_value:.3f} (HIGH - Consider removing)")

print("\nFeatures that are SAFE to use:")
safe_features = [f for f in feature_cols[1:] 
                 if abs(correlation_matrix.loc['calculated_capacity', f]) < 0.7
                 and f != 'calculated_capacity']
for feature in safe_features:
    corr_val = correlation_matrix.loc['calculated_capacity', feature]
    print(f"   - {feature} (correlation: {corr_val:.3f})")

#### **4.5 Process All Clean Cycles with Interpolation**

In [ ]:
def create_sequence_dataset(features_df: pd.DataFrame,
                          n_points: int = 100,
                          include_features: List[str] = ['voltage', 'current', 'temperature'],
                          exclude_additional: List[str] = ['energy_wh', 'ambient_temperature']) -> Dict:
    """
    Create dataset with interpolated sequences for deep learning
    """
    # Define all possible additional features
    all_additional_features = ['ambient_temperature', 'duration_hours', 'energy_wh',
                            'voltage_drop', 'temperature_mean', 'temperature_range',
                            'Re_mapped', 'Rct_mapped', 'impedance_time_diff']
        
    # Filter out excluded features
    use_features = [f for f in all_additional_features if f not in exclude_additional]
    
    # Only keep features that exist in the dataframe
    use_features = [f for f in use_features if f in features_df.columns]
    
    print(f"Including additional features: {use_features}")
    
    sequences = {feat: [] for feat in include_features}
    capacities = []
    cycle_numbers = []
    battery_ids = []
    additional_features = []

    print(f"Processing {len(features_df)} cycles for sequence extraction...")

    for idx, row in tqdm(features_df.iterrows(), total=len(features_df), desc="Extracting sequences"):
        # Load cycle data
        cycle_data = load_cycle_data(row['filename'])

        # Interpolate profile (will handle different lengths due to voltage cutoff)
        interpolated = interpolate_discharge_profile(cycle_data, n_points=n_points)

        if interpolated is not None:
            # Add sequences
            for feat in include_features:
                sequences[feat].append(interpolated[feat])

            # Add target (capacity)
            capacities.append(row['calculated_capacity'])

            # Add metadata
            cycle_numbers.append(row['cycle_number'])
            battery_ids.append(row['battery_id'])

            # Add additional features
            feature_values = []
            for feat_name in use_features:
                value = row.get(feat_name, np.nan)
                # Handle NaN values
                if pd.isna(value):
                    if feat_name in ['Re', 'Rct']:
                        value = 0.0  # Or use mean imputation
                    else:
                        value = 0.0
                feature_values.append(value)
            
            additional_features.append(feature_values)

    # Convert to numpy arrays
    dataset = {
        'sequences': {feat: np.array(sequences[feat]) for feat in include_features},
        'capacities': np.array(capacities),
        'cycle_numbers': np.array(cycle_numbers),
        'battery_ids': np.array(battery_ids),
        'additional_features': np.array(additional_features),
        'feature_names': use_features
    }

    print(f"\nDataset created successfully!")
    print(f"Features used: {dataset['feature_names']}")

    return dataset

# Create the sequence dataset
sequence_dataset = create_sequence_dataset(discharge_features_clean)

#### **4.6 Visualize Battery Degradation with Interpolated Profiles**

In [ ]:
# Select one battery to visualize degradation over time
battery_id = 'B0005'  # This showed good degradation pattern
battery_mask = sequence_dataset['battery_ids'] == battery_id

battery_voltages = sequence_dataset['sequences']['voltage'][battery_mask]
battery_capacities = sequence_dataset['capacities'][battery_mask]
battery_cycles = sequence_dataset['cycle_numbers'][battery_mask]

# Select cycles to visualize (early, middle, late)
n_cycles = len(battery_cycles)
cycle_indices = [0, n_cycles//4, n_cycles//2, 3*n_cycles//4, n_cycles-1]

plt.figure(figsize=(12, 6))

# Voltage profiles
plt.subplot(1, 2, 1)
colors = plt.cm.viridis(np.linspace(0, 1, len(cycle_indices)))
for i, idx in enumerate(cycle_indices):
    plt.plot(battery_voltages[idx], color=colors[i],
             label=f'Cycle {battery_cycles[idx]} (Cap: {battery_capacities[idx]:.2f} Ah)')
plt.xlabel('Normalized Time Points')
plt.ylabel('Voltage (V)')
plt.title(f'Battery {battery_id} - Voltage Profile Evolution')
plt.legend()
plt.grid(True, alpha=0.3)

# Capacity degradation
plt.subplot(1, 2, 2)
plt.plot(battery_cycles, battery_capacities, 'o-', markersize=4)
plt.scatter(battery_cycles[cycle_indices], battery_capacities[cycle_indices],
           color='red', s=100, zorder=5, label='Visualized cycles')
plt.xlabel('Cycle Number')
plt.ylabel('Capacity (Ah)')
plt.title(f'Battery {battery_id} - Capacity Degradation')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'battery_performance_visualization.png'))
plt.show()

#### **4.7 Save Processed Dataset**

In [ ]:
# Save the complete dataset
PROC_DATA_DIR = os.path.join(EXPORT_DIR,'processed_data')
os.makedirs(PROC_DATA_DIR, exist_ok=True)

output_file = os.path.join(PROC_DATA_DIR, "sequence_dataset_interpolated.pkl")
with open(output_file, 'wb') as f:
    pickle.dump(sequence_dataset, f)
print(f"Sequence dataset saved to: {output_file}")

# Also save the clean features dataframe
discharge_features_clean.to_csv(
    os.path.join(PROC_DATA_DIR, "discharge_features_clean.csv"),
    index=False
)
print(f"Clean features saved to: {os.path.join(PROC_DATA_DIR, 'discharge_features_clean.csv')}")

### **5. Deep Learning Data Preparation**

#### **5.1 Import Deep Learning Libraries**

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam

#### **5.2 Create Train/Test Split by Battery**

In [ ]:
def create_battery_based_split(dataset: Dict, test_size: float = 0.2, random_state: int = 42):
    """
    Split data by battery ID to prevent data leakage
    """
    unique_batteries = np.unique(dataset['battery_ids'])

    # Split batteries into train and test
    train_batteries, test_batteries = train_test_split(
        unique_batteries,
        test_size=test_size,
        random_state=random_state
    )

    # Create masks
    train_mask = np.isin(dataset['battery_ids'], train_batteries)
    test_mask = np.isin(dataset['battery_ids'], test_batteries)

    # Split data
    train_data = {
        'sequences': {key: val[train_mask] for key, val in dataset['sequences'].items()},
        'capacities': dataset['capacities'][train_mask],
        'cycle_numbers': dataset['cycle_numbers'][train_mask],
        'battery_ids': dataset['battery_ids'][train_mask],
        'additional_features': dataset['additional_features'][train_mask]
    }

    test_data = {
        'sequences': {key: val[test_mask] for key, val in dataset['sequences'].items()},
        'capacities': dataset['capacities'][test_mask],
        'cycle_numbers': dataset['cycle_numbers'][test_mask],
        'battery_ids': dataset['battery_ids'][test_mask],
        'additional_features': dataset['additional_features'][test_mask]
    }

    print(f"Train batteries ({len(train_batteries)}): {sorted(train_batteries)}")
    print(f"Test batteries ({len(test_batteries)}): {sorted(test_batteries)}")
    print(f"\nTrain samples: {len(train_data['capacities'])}")
    print(f"Test samples: {len(test_data['capacities'])}")

    return train_data, test_data, train_batteries, test_batteries

# Create train/test split
train_data, test_data, train_batteries, test_batteries = create_battery_based_split(sequence_dataset)

#### **5.3 Data Normalization**

In [ ]:
def normalize_sequences(train_data: Dict, test_data: Dict) -> Tuple[Dict, Dict, Dict]:
    """
    Normalize sequences and additional features
    """
    scalers = {}

    # Normalize sequences (voltage, current, temperature)
    for seq_type in train_data['sequences'].keys():
        scaler = MinMaxScaler()

        # Fit on training data
        train_seq = train_data['sequences'][seq_type]
        train_seq_reshaped = train_seq.reshape(-1, 1)
        scaler.fit(train_seq_reshaped)

        # Transform both train and test
        train_data['sequences'][seq_type] = scaler.transform(
            train_seq.reshape(-1, 1)
        ).reshape(train_seq.shape)

        test_seq = test_data['sequences'][seq_type]
        test_data['sequences'][seq_type] = scaler.transform(
            test_seq.reshape(-1, 1)
        ).reshape(test_seq.shape)

        scalers[f'seq_{seq_type}'] = scaler

    # Normalize additional features
    feature_scaler = StandardScaler()
    train_data['additional_features'] = feature_scaler.fit_transform(
        train_data['additional_features']
    )
    test_data['additional_features'] = feature_scaler.transform(
        test_data['additional_features']
    )
    scalers['additional_features'] = feature_scaler

    # Normalize capacities (target)
    capacity_scaler = MinMaxScaler()
    train_capacities = train_data['capacities'].reshape(-1, 1)
    test_capacities = test_data['capacities'].reshape(-1, 1)

    train_data['capacities_normalized'] = capacity_scaler.fit_transform(train_capacities).flatten()
    test_data['capacities_normalized'] = capacity_scaler.transform(test_capacities).flatten()
    scalers['capacity'] = capacity_scaler

    print("Data normalization completed")
    print(f"Voltage range: [{train_data['sequences']['voltage'].min():.3f}, {train_data['sequences']['voltage'].max():.3f}]")
    print(f"Current range: [{train_data['sequences']['current'].min():.3f}, {train_data['sequences']['current'].max():.3f}]")
    print(f"Temperature range: [{train_data['sequences']['temperature'].min():.3f}, {train_data['sequences']['temperature'].max():.3f}]")

    return train_data, test_data, scalers

# Normalize data
train_data, test_data, scalers = normalize_sequences(train_data, test_data)

#### **5.4 Prepare Input Data for Models**

In [ ]:
def prepare_model_inputs(data: Dict, sequence_features: List[str] = ['voltage', 'current', 'temperature']):
    """
    Prepare inputs for deep learning models
    """
    # Stack sequence features
    sequence_list = [data['sequences'][feat] for feat in sequence_features]
    sequences_stacked = np.stack(sequence_list, axis=2)  # Shape: (samples, time_steps, features)

    # Additional features
    additional_features = data['additional_features']

    # Targets
    targets = data['capacities_normalized']

    return sequences_stacked, additional_features, targets

# Prepare training data
X_seq_train, X_add_train, y_train = prepare_model_inputs(train_data)
X_seq_test, X_add_test, y_test = prepare_model_inputs(test_data)

print(f"Training sequences shape: {X_seq_train.shape}")
print(f"Training additional features shape: {X_add_train.shape}")
print(f"Training targets shape: {y_train.shape}")
print(f"\nTest sequences shape: {X_seq_test.shape}")
print(f"Test additional features shape: {X_add_test.shape}")
print(f"Test targets shape: {y_test.shape}")

print("\nData Quality Validation:")
print("=" * 40)

# Current variation check
current_std_per_sample = np.std(X_seq_train[:, :, 1], axis=1)
print(f"Current variation per sample - Mean STD: {np.mean(current_std_per_sample):.6f}")
print(f"Current variation range: [{np.min(current_std_per_sample):.6f}, {np.max(current_std_per_sample):.6f}]")

if np.mean(current_std_per_sample) < 0.01:
    print("WARNING: Current profiles show minimal variation - consider excluding")
else:
    print("Current profiles show adequate variation")

# Additional validation
voltage_std_per_sample = np.std(X_seq_train[:, :, 0], axis=1)
temp_std_per_sample = np.std(X_seq_train[:, :, 2], axis=1)

print(f"Voltage variation per sample - Mean STD: {np.mean(voltage_std_per_sample):.6f}")
print(f"Temperature variation per sample - Mean STD: {np.mean(temp_std_per_sample):.6f}")

# Check for any completely flat profiles
flat_current = np.sum(current_std_per_sample < 0.001)
flat_voltage = np.sum(voltage_std_per_sample < 0.001)
flat_temp = np.sum(temp_std_per_sample < 0.001)

print(f"\nFlat profiles detected:")
print(f"Current: {flat_current}/{len(current_std_per_sample)} samples")
print(f"Voltage: {flat_voltage}/{len(voltage_std_per_sample)} samples") 
print(f"Temperature: {flat_temp}/{len(temp_std_per_sample)} samples")

#### **5.5 Create Data Visualization for Model Input**

In [ ]:
# Visualize sample normalized sequences
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Select random samples
sample_indices = np.random.choice(len(X_seq_train), 3, replace=False)

for i, idx in enumerate(sample_indices):
    # Voltage
    axes[0, i].plot(X_seq_train[idx, :, 0])
    axes[0, i].set_title(f'Sample {i+1} - Voltage (Normalized)')
    axes[0, i].set_xlabel('Time Points')
    axes[0, i].set_ylabel('Normalized Voltage')
    axes[0, i].grid(True, alpha=0.3)

    # Current
    axes[1, i].plot(X_seq_train[idx, :, 1], color='orange')
    axes[1, i].set_title(f'Sample {i+1} - Current (Normalized)')
    axes[1, i].set_xlabel('Time Points')
    axes[1, i].set_ylabel('Normalized Current')
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'sample_normalized_sequences.png'))
plt.show()

# Show capacity distribution
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(train_data['capacities'], bins=30, alpha=0.7, label='Train')
plt.hist(test_data['capacities'], bins=30, alpha=0.7, label='Test')
plt.xlabel('Capacity (Ah)')
plt.ylabel('Count')
plt.title('Capacity Distribution')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(y_train, bins=30, alpha=0.7, label='Train (Normalized)')
plt.hist(y_test, bins=30, alpha=0.7, label='Test (Normalized)')
plt.xlabel('Normalized Capacity')
plt.ylabel('Count')
plt.title('Normalized Capacity Distribution')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'capacity_distribution.png'))
plt.show()

#### **5.6 Save Prepared Data**

In [ ]:
# Save prepared data for models
prepared_data = {
    'X_seq_train': X_seq_train,
    'X_add_train': X_add_train,
    'y_train': y_train,
    'X_seq_test': X_seq_test,
    'X_add_test': X_add_test,
    'y_test': y_test,
    'train_data': train_data,
    'test_data': test_data,
    'scalers': scalers,
    'train_batteries': train_batteries,
    'test_batteries': test_batteries
}

output_file = os.path.join(PROC_DATA_DIR, "prepared_dl_data.pkl")
with open(output_file, 'wb') as f:
    pickle.dump(prepared_data, f)
print(f"Prepared data saved to: {output_file}")

### **6. Deep Learning Model Implementation**

#### **6.1 Import Additional Libraries**

In [ ]:
import time
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.layers import Dense, Dropout, Input, Concatenate, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

#### **6.2 Define Training History Plotting Function**

In [ ]:
def plot_training_history(history, save_path=None):
    """
    Plot training history similar to W&B interface
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss plot
    axes[0,0].plot(history.history['loss'], label='Training Loss', color='blue')
    axes[0,0].plot(history.history['val_loss'], label='Validation Loss', color='red')
    axes[0,0].set_title('Model Loss')
    axes[0,0].set_xlabel('Epoch')
    axes[0,0].set_ylabel('Loss (MSE)')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # MAE plot
    axes[0,1].plot(history.history['mae'], label='Training MAE', color='blue')
    axes[0,1].plot(history.history['val_mae'], label='Validation MAE', color='red')
    axes[0,1].set_title('Model MAE')
    axes[0,1].set_xlabel('Epoch')
    axes[0,1].set_ylabel('MAE')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Learning rate (if using scheduler)
    if 'lr' in history.history:
        axes[1,0].plot(history.history['lr'], label='Learning Rate', color='green')
        axes[1,0].set_title('Learning Rate')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Learning Rate')
        axes[1,0].set_yscale('log')
        axes[1,0].legend()
        axes[1,0].grid(True, alpha=0.3)
    else:
        axes[1,0].text(0.5, 0.5, 'No LR Schedule', ha='center', va='center', transform=axes[1,0].transAxes)
        axes[1,0].set_title('Learning Rate')
    
    # Training vs Validation comparison
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]
    final_train_mae = history.history['mae'][-1]
    final_val_mae = history.history['val_mae'][-1]
    
    metrics = ['Loss', 'MAE']
    train_values = [final_train_loss, final_train_mae]
    val_values = [final_val_loss, final_val_mae]
    
    x = range(len(metrics))
    width = 0.35
    
    axes[1,1].bar([i - width/2 for i in x], train_values, width, label='Training', color='blue', alpha=0.7)
    axes[1,1].bar([i + width/2 for i in x], val_values, width, label='Validation', color='red', alpha=0.7)
    axes[1,1].set_title('Final Metrics Comparison')
    axes[1,1].set_ylabel('Value')
    axes[1,1].set_xticks(x)
    axes[1,1].set_xticklabels(metrics)
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print final metrics
    print(f"\nTraining Summary:")
    print(f"{'='*50}")
    print(f"Final Training Loss: {final_train_loss:.6f}")
    print(f"Final Validation Loss: {final_val_loss:.6f}")
    print(f"Final Training MAE: {final_train_mae:.6f}")
    print(f"Final Validation MAE: {final_val_mae:.6f}")
    print(f"Overfitting Check: {final_val_loss/final_train_loss:.2f}x")


#### **6.3 LSTM (Long-Short Term Memory) Implementation**

In [ ]:
def build_lstm_with_additional_features(sequence_shape, additional_features_shape,
                                lstm_units=50, dense_units=25, dropout_rate=0.2):
    """
    Build LSTM model with additional features
    """
    # Sequence input
    sequence_input = Input(shape=sequence_shape, name='sequence_input')

    # LSTM layers
    x = LSTM(lstm_units, return_sequences=True)(sequence_input)
    x = Dropout(dropout_rate)(x)
    x = LSTM(lstm_units)(x)
    x = Dropout(dropout_rate)(x)

    # Additional features input
    additional_input = Input(shape=(additional_features_shape,), name='additional_input')

    # Combine LSTM output with additional features
    combined = Concatenate()([x, additional_input])

    # Dense layers
    x = Dense(dense_units, activation='relu')(combined)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    # Output layer
    output = Dense(1, activation='linear', name='capacity_output')(x)

    # Create model
    model = models.Model(
        inputs=[sequence_input, additional_input],
        outputs=output
    )

    return model

# Build LSTM model
lstm_model = build_lstm_with_additional_features(
    sequence_shape=(X_seq_train.shape[1], X_seq_train.shape[2]),
    additional_features_shape=X_add_train.shape[1]
)

lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("\nLSTM Model Summary:")
lstm_model.summary() 

#### **6.4 TCN (Temporal Convolutional Network) Implementation**

In [ ]:
from tcn import TCN

def build_tcn_model(sequence_shape, additional_features_shape,
                   tcn_filters=256, tcn_kernel_size=7, tcn_dilations=[1, 2, 4, 8, 16, 32, 64],
                   dense_units=[512, 256, 128], dropout_rate=0.2):
    """
    Build TCN model with additional features
    """
    # Sequence input
    sequence_input = Input(shape=sequence_shape, name='sequence_input')

    # TCN layers
    tcn_out = TCN(
        nb_filters=tcn_filters,
        kernel_size=tcn_kernel_size,
        dilations=tcn_dilations,
        return_sequences=False,
        activation='relu',
        dropout_rate=dropout_rate
    )(sequence_input)

    # Additional features input
    additional_input = Input(shape=(additional_features_shape,), name='additional_input')

    # Combine TCN output with additional features
    combined = Concatenate()([tcn_out, additional_input])

    # Dense layers
    x = combined
    for units in dense_units:
        x = Dense(units, activation='relu')(x)
        x = BatchNormalization()(x)
        x = Dropout(dropout_rate)(x)

    # Output layer
    output = Dense(1, activation='linear', name='capacity_output')(x)

    # Create model
    model = models.Model(
        inputs=[sequence_input, additional_input],
        outputs=output
    )

    return model

# Build TCN model
tcn_model = build_tcn_model(
    sequence_shape=(X_seq_train.shape[1], X_seq_train.shape[2]),
    additional_features_shape=X_add_train.shape[1]
)

tcn_model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='mse',
    metrics=['mae']
)

print("TCN Model Summary:")
tcn_model.summary()

#### **6.5 BiLSTM (Bidirectional LSTM) Implementation**

In [ ]:
from tensorflow.keras.layers import LSTM, Bidirectional

def build_bilstm_model(sequence_shape, additional_features_shape,
                      lstm_units=[64, 32], dense_units=[128, 64],
                      dropout_rate=0.2):
    """
    Build Bidirectional LSTM model with additional features
    """
    # Sequence input
    sequence_input = Input(shape=sequence_shape, name='sequence_input')

    # BiLSTM layers
    x = sequence_input
    for i, units in enumerate(lstm_units):
        return_sequences = (i < len(lstm_units) - 1)
        x = Bidirectional(
            LSTM(units, return_sequences=return_sequences, dropout=dropout_rate)
        )(x)
        if return_sequences:
            x = BatchNormalization()(x)

    # Additional features input
    additional_input = Input(shape=(additional_features_shape,), name='additional_input')

    # Combine LSTM output with additional features
    combined = Concatenate()([x, additional_input])

    # Dense layers
    x = combined
    for units in dense_units:
        x = Dense(units, activation='relu')(x)
        x = BatchNormalization()(x)
        x = Dropout(dropout_rate)(x)

    # Output layer
    output = Dense(1, activation='linear', name='capacity_output')(x)

    # Create model
    model = models.Model(
        inputs=[sequence_input, additional_input],
        outputs=output
    )

    return model

# Build BiLSTM model
bilstm_model = build_bilstm_model(
    sequence_shape=(X_seq_train.shape[1], X_seq_train.shape[2]),
    additional_features_shape=X_add_train.shape[1]
)

bilstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("\nBiLSTM Model Summary:")
bilstm_model.summary()

#### **6.6 Define Training Callbacks**

In [ ]:
def create_callbacks(model_name, patience_early=20, patience_lr=10):
    """
    Create callbacks with model-specific parameters
    """
    MODEL_DIR = os.path.join(EXPORT_DIR, 'models')
    os.makedirs(MODEL_DIR, exist_ok=True)
    
    callbacks_list = [
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=patience_early,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=patience_lr,
            min_lr=1e-6,
            verbose=1
        ),
        callbacks.ModelCheckpoint(
            filepath=os.path.join(MODEL_DIR, f'{model_name}_best.h5'), 
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        )
    ]
    return callbacks_list

# Use different parameters for each model
lstm_callbacks = create_callbacks('lstm_model', patience_early=20, patience_lr=10)
bilstm_callbacks = create_callbacks('bilstm_model', patience_early=20, patience_lr=10)
tcn_callbacks = create_callbacks('tcn_model', patience_early=30, patience_lr=15) 

def format_time(seconds):
    minutes = int(seconds // 60)
    remaining_seconds = int(seconds % 60)
    return f"{minutes}m {remaining_seconds}s"

#### **6.7 Create Validation Split**

In [ ]:
def create_fixed_validation_split(train_data: Dict, train_batteries: np.ndarray):
    """
    Create FIXED validation split to ensure reproducibility
    """
    val_batteries_fixed = ['B0007', 'B0032', 'B0041', 'B0043', 'B0055']
    train_batteries_fixed = [b for b in train_batteries if b not in val_batteries_fixed]
    
    # Create masks for the split
    train_mask = np.isin(train_data['battery_ids'], train_batteries_fixed)
    val_mask = np.isin(train_data['battery_ids'], val_batteries_fixed)
    
    print(f"Train batteries for training ({len(train_batteries_fixed)}): {sorted(train_batteries_fixed)}")
    print(f"Validation batteries ({len(val_batteries_fixed)}): {sorted(val_batteries_fixed)}")
    
    return train_mask, val_mask, train_batteries_fixed, val_batteries_fixed

# Create FIXED battery-based validation split
train_mask, val_mask, train_batteries_final, val_batteries_final = create_fixed_validation_split(
    train_data, train_batteries
)

# Apply the masks to create validation split
X_seq_train_split = X_seq_train[train_mask]
X_seq_val_split = X_seq_train[val_mask]
X_add_train_split = X_add_train[train_mask]
X_add_val_split = X_add_train[val_mask]
y_train_split = y_train[train_mask]
y_val_split = y_train[val_mask]

print(f"\nTraining samples: {len(X_seq_train_split)}")
print(f"Validation samples: {len(X_seq_val_split)}")

# Verify no battery overlap
train_battery_check = train_data['battery_ids'][train_mask]
val_battery_check = train_data['battery_ids'][val_mask]
overlap = set(train_battery_check) & set(val_battery_check)
print(f"Battery overlap between train and val: {len(overlap)} (should be 0)")

In [ ]:
# Verify the features being used
print("Final Feature Verification")
print("="*50)
print(f"Number of sequence features: {X_seq_train.shape[2]}")
print(f"Number of additional features: {X_add_train.shape[1]}")

# Print remaining feature names if available
if 'feature_names' in sequence_dataset:
    remaining_features = [f for f in sequence_dataset['feature_names'] 
                         if f not in ['energy_wh', 'ambient_temperature']]
    print(f"Remaining features: {remaining_features}")

#### **6.8 Train LSTM Model**

In [ ]:
lstm_gpu_used, gpu_total = get_gpu_memory_usage()
lstm_start_time = time.time()

# Train LSTM model
lstm_history = lstm_model.fit(
    [X_seq_train_split, X_add_train_split],
    y_train_split,
    validation_data=([X_seq_val_split, X_add_val_split], y_val_split),
    epochs=100,
    batch_size=32,
    callbacks=lstm_callbacks,
    verbose=1
)

# Measure time after training
lstm_end_time = time.time()
lstm_training_time = lstm_end_time - lstm_start_time

# Print summary
print("=" * 50)
print(f"LSTM training time: {format_time(lstm_training_time)}")
if lstm_gpu_used is not None and gpu_total is not None:
    print(f"LSTM GPU resource used: {lstm_gpu_used: .2f} GB / {gpu_total: .2f} GB")

# Plot training history
plot_training_history(lstm_history, save_path=os.path.join(PLOT_DIR, 'lstm_training_history.png'))

#### **6.9 Train TCN Model**

In [ ]:
# Train TCN model
tcn_start_time = time.time()
tcn_gpu_used, gpu_total = get_gpu_memory_usage()

tcn_history = tcn_model.fit(
    [X_seq_train_split, X_add_train_split],
    y_train_split,
    validation_data=([X_seq_val_split, X_add_val_split], y_val_split),
    epochs=100,
    batch_size=32,
    callbacks=tcn_callbacks,
    verbose=1
)

# Measure time after training
tcn_end_time = time.time()
tcn_training_time = tcn_end_time - tcn_start_time

# Print summary
print("=" * 50)
print(f"TCN training time: {format_time(tcn_training_time)}")
if tcn_gpu_used is not None and gpu_total is not None:
    print(f"TCN GPU resource used: {tcn_gpu_used:.2f} GB / {gpu_total:.2f} GB")

# Plot training history
plot_training_history(tcn_history, save_path=os.path.join(PLOT_DIR, 'tcn_training_history.png'))

#### **6.10 Train BiLSTM Model**

In [ ]:
# Train BiLSTM model
bilstm_start_time = time.time()
bilstm_gpu_used, gpu_total = get_gpu_memory_usage()

bilstm_history = bilstm_model.fit(
    [X_seq_train_split, X_add_train_split],
    y_train_split,
    validation_data=([X_seq_val_split, X_add_val_split], y_val_split),
    epochs=100,
    batch_size=32,
    callbacks=bilstm_callbacks,
    verbose=1
)

# Measure time after training
bilstm_end_time = time.time()
bilstm_training_time = bilstm_end_time - bilstm_start_time

print("=" * 50)
print(f"BiLSTM training time: {format_time(bilstm_training_time)}")
if bilstm_gpu_used is not None and gpu_total is not None:
    print(f"BiLSTM GPU resource used: {bilstm_gpu_used: .2f} GB / {gpu_total: .2f} GB")

# Plot training history
plot_training_history(bilstm_history, save_path=os.path.join(PLOT_DIR, 'bilstm_training_history.png'))

### **7. Model Evaluation and Comparison**


#### **7.1 Define Evaluation Functions**


In [ ]:
def evaluate_model(model, X_seq, X_add, y_true, scalers, model_name):
    """
    Evaluate model and return predictions and metrics
    """
    # Get predictions
    y_pred_normalized = model.predict([X_seq, X_add])

    # Denormalize predictions
    y_pred = scalers['capacity'].inverse_transform(y_pred_normalized.reshape(-1, 1)).flatten()
    y_true_denorm = scalers['capacity'].inverse_transform(y_true.reshape(-1, 1)).flatten()

    # Calculate metrics
    mae = mean_absolute_error(y_true_denorm, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true_denorm, y_pred))
    r2 = r2_score(y_true_denorm, y_pred)

    # Calculate percentage errors
    mape = np.mean(np.abs((y_true_denorm - y_pred) / y_true_denorm)) * 100

    # Statistical significance testing
    from scipy import stats

    def statistical_comparison(errors1, errors2, model1_name, model2_name):
        """Perform paired t-test for statistical significance"""
        t_stat, p_value = stats.ttest_rel(errors1, errors2)
        return {
            'comparison': f'{model1_name} vs {model2_name}',
            't_statistic': t_stat,
            'p_value': p_value,
            'significant': p_value < 0.05
        }

    metrics = {
        'Model': model_name,
        'MAE (Ah)': mae,
        'RMSE (Ah)': rmse,
        'R²': r2,
        'MAPE (%)': mape
    }

    return y_pred, y_true_denorm, metrics

# Evaluate both models on test set
lstm_pred, y_test_true, lstm_metrics = evaluate_model(
    lstm_model, X_seq_test, X_add_test, y_test, scalers, 'LSTM'
)

tcn_pred, y_test_true, tcn_metrics = evaluate_model(
    tcn_model, X_seq_test, X_add_test, y_test, scalers, 'TCN'
)

bilstm_pred, _, bilstm_metrics = evaluate_model(
    bilstm_model, X_seq_test, X_add_test, y_test, scalers, 'BiLSTM'
)

#### **7.2 Create Prediction vs Actual Plots for DL models**

In [ ]:
# Create prediction plots with confidence intervals
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
models_data = [
    (tcn_pred, 'TCN', 'blue', tcn_metrics),
    (bilstm_pred, 'BiLSTM', 'green', bilstm_metrics), 
    (lstm_pred, 'LSTM', 'orange', lstm_metrics)
]

for i, (predictions, model_name, color, metrics) in enumerate(models_data):
    # Calculate prediction intervals
    residuals = predictions - y_test_true
    std_residual = np.std(residuals)
    
    # Colored scatter points for each model
    axes[i].scatter(y_test_true, predictions, alpha=0.6, s=30, color=color, edgecolors='black', linewidth=0.5)
    
    # Perfect prediction line
    min_val, max_val = y_test_true.min(), y_test_true.max()
    axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, 
                label='Perfect Prediction')
    
    # Confidence bands
    axes[i].fill_between([min_val, max_val], 
                        [min_val - 2*std_residual, max_val - 2*std_residual],
                        [min_val + 2*std_residual, max_val + 2*std_residual],
                        alpha=0.2, color='gray', label='95% Confidence')
    
    axes[i].set_xlabel('Actual Capacity (Ah)', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Predicted Capacity (Ah)', fontsize=12, fontweight='bold')
    axes[i].set_title(f'{model_name}\nR² = {metrics["R²"]:.4f}, RMSE = {metrics["RMSE (Ah)"]:.3f} Ah', 
                     fontsize=14, fontweight='bold')
    axes[i].legend(fontsize=10)
    axes[i].grid(True, alpha=0.3, linestyle=':')
    
    # Add correlation coefficient annotation
    corr_coef = np.corrcoef(y_test_true, predictions)[0, 1]
    axes[i].annotate(f'ρ = {corr_coef:.4f}', xy=(0.05, 0.95), 
                    xycoords='axes fraction', fontsize=11,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'model_predictions_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

#### **7.3 Model Robustness Analysis**

In [ ]:
def analyze_model_robustness(predictions, true_values, model_name):
    """Analyze model robustness across different capacity ranges"""
    errors = np.abs(predictions - true_values)
    
    # Define capacity ranges
    capacity_ranges = [
        (true_values.min(), np.percentile(true_values, 25), 'Low (0-25%)'),
        (np.percentile(true_values, 25), np.percentile(true_values, 50), 'Medium-Low (25-50%)'),
        (np.percentile(true_values, 50), np.percentile(true_values, 75), 'Medium-High (50-75%)'),
        (np.percentile(true_values, 75), true_values.max(), 'High (75-100%)')
    ]
    
    robustness_data = []
    for min_cap, max_cap, range_name in capacity_ranges:
        mask = (true_values >= min_cap) & (true_values <= max_cap)
        if np.sum(mask) > 0:
            range_mae = np.mean(errors[mask])
            range_std = np.std(errors[mask])
            robustness_data.append({
                'Model': model_name,
                'Range': range_name,
                'MAE': range_mae,
                'Std': range_std,
                'Samples': np.sum(mask)
            })
    
    return pd.DataFrame(robustness_data)

# Analyze robustness for all models
robustness_results = []
for pred, name in [(tcn_pred, 'TCN'), (bilstm_pred, 'BiLSTM'), (lstm_pred, 'LSTM')]:
    robustness_results.append(analyze_model_robustness(pred, y_test_true, name))

robustness_df = pd.concat(robustness_results, ignore_index=True)
print("\nModel Robustness Analysis:")
display(robustness_df)

#### **7.4 Create Error Distribution Plots**

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# TCN Error Distribution
tcn_errors = tcn_pred - y_test_true
axes[0, 0].hist(tcn_errors, bins=30, alpha=0.7, color='blue', edgecolor='black')
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Prediction Error (Ah)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title(f'TCN - Error Distribution\nMean Error: {np.mean(tcn_errors):.4f} Ah')
axes[0, 0].grid(True, alpha=0.3)

# BiLSTM Error Distribution
bilstm_errors = bilstm_pred - y_test_true
axes[0, 1].hist(bilstm_errors, bins=30, alpha=0.7, color='green', edgecolor='black')
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Prediction Error (Ah)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title(f'BiLSTM - Error Distribution\nMean Error: {np.mean(bilstm_errors):.4f} Ah')
axes[0, 1].grid(True, alpha=0.3)

# LSTM Error Distribution
lstm_errors = lstm_pred - y_test_true
axes[0, 2].hist(lstm_errors, bins=30, alpha=0.7, color='orange', edgecolor='black')
axes[0, 2].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 2].set_xlabel('Prediction Error (Ah)')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].set_title(f'LSTM - Error Distribution\nMean Error: {np.mean(lstm_errors):.4f} Ah')
axes[0, 2].grid(True, alpha=0.3)

# TCN Residual Plot
axes[1, 0].scatter(tcn_pred, tcn_errors, alpha=0.5, s=30)
axes[1, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Predicted Capacity (Ah)')
axes[1, 0].set_ylabel('Residual (Ah)')
axes[1, 0].set_title('TCN - Residual Plot')
axes[1, 0].grid(True, alpha=0.3)

# BiLSTM Residual Plot
axes[1, 1].scatter(bilstm_pred, bilstm_errors, alpha=0.5, s=30, color='green')
axes[1, 1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Predicted Capacity (Ah)')
axes[1, 1].set_ylabel('Residual (Ah)')
axes[1, 1].set_title('BiLSTM - Residual Plot')
axes[1, 1].grid(True, alpha=0.3)

# LSTM Residual Plot
axes[1, 2].scatter(lstm_pred, lstm_errors, alpha=0.5, s=30, color='orange')
axes[1, 2].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 2].set_xlabel('Predicted Capacity (Ah)')
axes[1, 2].set_ylabel('Residual (Ah)')
axes[1, 2].set_title('LSTM - Residual Plot')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'model_error_analysis.png'))
plt.show()

#### **7.5 Per-Battery Performance Analysis**

In [ ]:
def analyze_battery_performance(model, X_seq, X_add, y_true, battery_ids, scalers, model_name):
    """
    Analyze model performance per battery
    """
    # Get predictions
    y_pred_normalized = model.predict([X_seq, X_add])
    y_pred = scalers['capacity'].inverse_transform(y_pred_normalized.reshape(-1, 1)).flatten()
    y_true_denorm = scalers['capacity'].inverse_transform(y_true.reshape(-1, 1)).flatten()

    # Calculate metrics per battery
    battery_metrics = []
    for battery in np.unique(battery_ids):
        mask = battery_ids == battery

        if np.sum(mask) > 0:
            mae = mean_absolute_error(y_true_denorm[mask], y_pred[mask])
            rmse = np.sqrt(mean_squared_error(y_true_denorm[mask], y_pred[mask]))
            r2 = r2_score(y_true_denorm[mask], y_pred[mask])

            battery_metrics.append({
                'Battery': battery,
                'Samples': np.sum(mask),
                'MAE (Ah)': mae,
                'RMSE (Ah)': rmse,
                'R²': r2
            })

    return pd.DataFrame(battery_metrics)

def analyze_degradation_patterns(model, X_seq, X_add, y_true, battery_ids, cycle_numbers, scalers, model_name):
    """Analyze model performance across different degradation stages"""
    y_pred_normalized = model.predict([X_seq, X_add])
    y_pred = scalers['capacity'].inverse_transform(y_pred_normalized.reshape(-1, 1)).flatten()
    y_true_denorm = scalers['capacity'].inverse_transform(y_true.reshape(-1, 1)).flatten()
    
    degradation_analysis = []
    for battery in np.unique(battery_ids):
        mask = battery_ids == battery
        if np.sum(mask) < 10:  # Skip batteries with too few samples
            continue
            
        battery_cycles = cycle_numbers[mask]
        battery_true = y_true_denorm[mask]
        battery_pred = y_pred[mask]
        
        # Sort by cycle number to ensure proper ordering
        sort_idx = np.argsort(battery_cycles)
        battery_cycles = battery_cycles[sort_idx]
        battery_true = battery_true[sort_idx]
        battery_pred = battery_pred[sort_idx]
        
        # Calculate degradation rate (Ah per cycle)
        if len(battery_true) > 1:
            true_degradation_rate = (battery_true[-1] - battery_true[0]) / (battery_cycles[-1] - battery_cycles[0])
            pred_degradation_rate = (battery_pred[-1] - battery_pred[0]) / (battery_cycles[-1] - battery_cycles[0])
            
            # Calculate stage-wise performance
            n_samples = len(battery_true)
            early_end = n_samples // 3
            late_start = 2 * n_samples // 3
            
            early_stage_mae = mean_absolute_error(battery_true[:early_end], battery_pred[:early_end]) if early_end > 0 else np.nan
            late_stage_mae = mean_absolute_error(battery_true[late_start:], battery_pred[late_start:]) if late_start < n_samples else np.nan
            
            degradation_analysis.append({
                'Battery': battery,
                'Model': model_name,
                'Total_Samples': n_samples,
                'Cycle_Range': f"{battery_cycles[0]:.0f}-{battery_cycles[-1]:.0f}",
                'True_Degradation_Rate': true_degradation_rate,
                'Pred_Degradation_Rate': pred_degradation_rate,
                'Degradation_Error': abs(true_degradation_rate - pred_degradation_rate),
                'Degradation_Error_Pct': abs(true_degradation_rate - pred_degradation_rate) / abs(true_degradation_rate) * 100 if true_degradation_rate != 0 else np.nan,
                'Early_Stage_MAE': early_stage_mae,
                'Late_Stage_MAE': late_stage_mae,
                'Overall_MAE': mean_absolute_error(battery_true, battery_pred)
            })
    
    return pd.DataFrame(degradation_analysis)

# Analyze per-battery performance (existing function)
tcn_battery_perf = analyze_battery_performance(
    tcn_model, X_seq_test, X_add_test, y_test,
    test_data['battery_ids'], scalers, 'TCN'
)

bilstm_battery_perf = analyze_battery_performance(
    bilstm_model, X_seq_test, X_add_test, y_test,
    test_data['battery_ids'], scalers, 'BiLSTM'
)

lstm_battery_perf = analyze_battery_performance(
    lstm_model, X_seq_test, X_add_test, y_test,
    test_data['battery_ids'], scalers, 'LSTM'
)

print("\nTCN - Per Battery Performance:")
display(tcn_battery_perf.sort_values('R²', ascending=False))

print("\nBiLSTM - Per Battery Performance:")
display(bilstm_battery_perf.sort_values('R²', ascending=False))

print("\nLSTM - Per Battery Performance:")
display(lstm_battery_perf.sort_values('R²', ascending=False))

# Now analyze degradation patterns
print("\n" + "="*80)
print("DEGRADATION PATTERN ANALYSIS")
print("="*80)

tcn_degradation = analyze_degradation_patterns(
    tcn_model, X_seq_test, X_add_test, y_test,
    test_data['battery_ids'], test_data['cycle_numbers'], scalers, 'TCN'
)

bilstm_degradation = analyze_degradation_patterns(
    bilstm_model, X_seq_test, X_add_test, y_test,
    test_data['battery_ids'], test_data['cycle_numbers'], scalers, 'BiLSTM'
)

lstm_degradation = analyze_degradation_patterns(
    lstm_model, X_seq_test, X_add_test, y_test,
    test_data['battery_ids'], test_data['cycle_numbers'], scalers, 'LSTM'
)

# Combine all degradation analyses
all_degradation = pd.concat([tcn_degradation, bilstm_degradation, lstm_degradation], ignore_index=True)

print("\nDegradation Rate Analysis:")
display(all_degradation.style.format({
    'True_Degradation_Rate': '{:.6f}',
    'Pred_Degradation_Rate': '{:.6f}',
    'Degradation_Error': '{:.6f}',
    'Degradation_Error_Pct': '{:.2f}%',
    'Early_Stage_MAE': '{:.4f}',
    'Late_Stage_MAE': '{:.4f}',
    'Overall_MAE': '{:.4f}'
}))

# Summary statistics for degradation analysis
print("\nDegradation Analysis Summary by Model:")
degradation_summary = all_degradation.groupby('Model').agg({
    'Degradation_Error': ['mean', 'std'],
    'Degradation_Error_Pct': ['mean', 'std'],
    'Early_Stage_MAE': ['mean', 'std'],
    'Late_Stage_MAE': ['mean', 'std']
}).round(4)

display(degradation_summary)

#### **7.6 Visualize Predictions for Sample Batteries**

In [ ]:
# Select two test batteries to visualize
sample_batteries = test_batteries[1:3]

fig, axes = plt.subplots(2, 3, figsize=(14, 10))

for i, battery_id in enumerate(sample_batteries):
    # Get data for this battery
    battery_mask = test_data['battery_ids'] == battery_id
    battery_cycles = test_data['cycle_numbers'][battery_mask]
    battery_true = scalers['capacity'].inverse_transform(
        test_data['capacities_normalized'][battery_mask].reshape(-1, 1)
    ).flatten()

    # Get predictions
    battery_tcn_pred = scalers['capacity'].inverse_transform(
        tcn_model.predict([X_seq_test[battery_mask], X_add_test[battery_mask]])
    ).flatten()

    battery_bilstm_pred = scalers['capacity'].inverse_transform(
        bilstm_model.predict([X_seq_test[battery_mask], X_add_test[battery_mask]])
    ).flatten()

    battery_lstm_pred = scalers['capacity'].inverse_transform(
        lstm_model.predict([X_seq_test[battery_mask], X_add_test[battery_mask]])
    ).flatten()

    # TCN Plot
    axes[i, 0].plot(battery_cycles, battery_true, 'o-', label='Actual', markersize=3, color='grey')
    axes[i, 0].plot(battery_cycles, battery_tcn_pred, 'o-', label='TCN Predicted', markersize=4, color='blue')
    axes[i, 0].set_xlabel('Cycle Number')
    axes[i, 0].set_ylabel('Capacity (Ah)')
    axes[i, 0].set_title(f'Battery {battery_id} - TCN Predictions')
    axes[i, 0].legend()
    axes[i, 0].grid(True, alpha=0.3)

    # BiLSTM Plot
    axes[i, 1].plot(battery_cycles, battery_true, 'o-', label='Actual', markersize=3, color='grey')
    axes[i, 1].plot(battery_cycles, battery_bilstm_pred, 'o-', label='BiLSTM Predicted', markersize=4, color='green')
    axes[i, 1].set_xlabel('Cycle Number')
    axes[i, 1].set_ylabel('Capacity (Ah)')
    axes[i, 1].set_title(f'Battery {battery_id} - BiLSTM Predictions')
    axes[i, 1].legend()
    axes[i, 1].grid(True, alpha=0.3)

    # LSTM Plot
    axes[i, 2].plot(battery_cycles, battery_true, 'o-', label='Actual', markersize=3, color='grey')
    axes[i, 2].plot(battery_cycles, battery_lstm_pred, 'o-', label='LSTM Predicted', markersize=4, color='orange')
    axes[i, 2].set_xlabel('Cycle Number')
    axes[i, 2].set_ylabel('Capacity (Ah)')
    axes[i, 2].set_title(f'Battery {battery_id} - LSTM Predictions')
    axes[i, 2].legend()
    axes[i, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'battery_performance_visualization.png'))
plt.show()

### **8. Compare Deep Learning Models Perform**

#### **8.1 Model Comparison**

In [ ]:
# Create comparison table for all models
baseline_comparison = pd.DataFrame([
    lstm_metrics,
    tcn_metrics,
    bilstm_metrics
])
baseline_comparison = baseline_comparison.sort_values('R²', ascending=True)

print("\nComplete Model Comparison:")
print("=" * 70)
display(baseline_comparison.style.format({
    'MAE (Ah)': '{:.4f}',
    'RMSE (Ah)': '{:.4f}',
    'R²': '{:.4f}',
    'MAPE (%)': '{:.2f}'
}))

# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = baseline_comparison['Model']
metrics_to_plot = ['MAE (Ah)', 'RMSE (Ah)', 'R²']

for i, metric in enumerate(metrics_to_plot):
    values = baseline_comparison[metric]
    colors = ['darkblue', 'orange', 'green', 'red', 'purple']

    axes[i].bar(models, values, color=colors)
    axes[i].set_title(f'Model Comparison - {metric}')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=45)

    # Add value labels on bars
    for j, v in enumerate(values):
        axes[i].text(j, v + 0.01 * max(values), f'{v:.3f}',
                    ha='center', va='bottom')

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'model_comparison.png'))
plt.show()

#### **8.2 Radar Plot for Model Comparison**

In [ ]:
from math import pi

def create_radar_chart(df):
    """Create IEEE-style radar chart for model comparison"""
    categories = ['MAE (Ah)', 'RMSE (Ah)', 'R²', 'MAPE (%)']
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
    
    # Normalize metrics for radar chart (higher = better)
    normalized_data = df.copy()
    for col in ['MAE (Ah)', 'RMSE (Ah)', 'MAPE (%)']:
        normalized_data[col] = 1 - (normalized_data[col] / normalized_data[col].max())
    
    angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
    angles += angles[:1]  # Complete the circle
    
    colors = ['blue', 'green', 'orange']  # Match your model colors
    for idx, (_, row) in enumerate(normalized_data.iterrows()):
        values = [row[cat] for cat in categories] + [row[categories[0]]]
        ax.plot(angles, values, 'o-', linewidth=3, label=row['Model'], color=colors[idx], markersize=8)
        ax.fill(angles, values, alpha=0.25, color=colors[idx])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12)
    ax.set_title('Model Performance Comparison\n(Radar Chart)', fontsize=18, fontweight='bold', pad=30)
    
    return fig

# Apply the radar chart to your comparison data
print("\nCreating Radar Chart for Model Comparison...")

# Use your existing baseline_comparison dataframe
radar_fig = create_radar_chart(baseline_comparison)
plt.savefig(os.path.join(PLOT_DIR, 'model_radar_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

# Also create a detailed comparison table
print("\nDetailed Model Performance Metrics:")
detailed_comparison = baseline_comparison.copy()
detailed_comparison['Rank_MAE'] = detailed_comparison['MAE (Ah)'].rank(ascending=True)
detailed_comparison['Rank_RMSE'] = detailed_comparison['RMSE (Ah)'].rank(ascending=True)
detailed_comparison['Rank_R²'] = detailed_comparison['R²'].rank(ascending=False)
detailed_comparison['Rank_MAPE'] = detailed_comparison['MAPE (%)'].rank(ascending=True)
detailed_comparison['Overall_Rank'] = (detailed_comparison['Rank_MAE'] + 
                                     detailed_comparison['Rank_RMSE'] + 
                                     detailed_comparison['Rank_R²'] + 
                                     detailed_comparison['Rank_MAPE']) / 4

detailed_comparison = detailed_comparison.sort_values('Overall_Rank')

display(detailed_comparison[['Model', 'MAE (Ah)', 'RMSE (Ah)', 'R²', 'MAPE (%)', 'Overall_Rank']].style.format({
    'MAE (Ah)': '{:.4f}',
    'RMSE (Ah)': '{:.4f}',
    'R²': '{:.4f}',
    'MAPE (%)': '{:.2f}',
    'Overall_Rank': '{:.2f}'
}))

# Print interpretation
print("\nRadar Chart Interpretation:")
print("- Larger area = Better overall performance")
print("- For MAE, RMSE, MAPE: Values closer to edge = Better (lower error)")
print("- For R²: Values closer to edge = Better (higher correlation)")
print(f"- Best performing model: {detailed_comparison.iloc[0]['Model']}")

#### **8.3 Cross-Validation Analysis**

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def time_series_cv_evaluation(models_dict, X_seq, X_add, y, scalers, n_splits=5):
    """Perform time-series cross-validation"""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    cv_results = []
    all_fold_results = {}  # Store all fold results by model
    
    for model_name, model in models_dict.items():
        print(f"Performing CV for {model_name}...")
        fold_scores = []
        
        for fold, (train_idx, val_idx) in enumerate(tscv.split(X_seq)):
            # Validation predictions
            val_pred_normalized = model.predict([X_seq[val_idx], X_add[val_idx]])
            val_true_normalized = y[val_idx]
            
            # Denormalize for proper metric calculation
            val_pred = scalers['capacity'].inverse_transform(val_pred_normalized.reshape(-1, 1)).flatten()
            val_true = scalers['capacity'].inverse_transform(val_true_normalized.reshape(-1, 1)).flatten()
            
            # Calculate metrics
            mae = mean_absolute_error(val_true, val_pred)
            rmse = np.sqrt(mean_squared_error(val_true, val_pred))
            r2 = r2_score(val_true, val_pred)
            mape = np.mean(np.abs((val_true - val_pred) / val_true)) * 100
            
            fold_scores.append({
                'Fold': fold+1, 
                'MAE': mae, 
                'RMSE': rmse, 
                'R²': r2,
                'MAPE': mape,
                'Samples': len(val_idx)
            })
        
        # Store fold results for this model
        all_fold_results[model_name] = fold_scores
        
        fold_df = pd.DataFrame(fold_scores)
        cv_results.append({
            'Model': model_name,
            'CV_MAE_Mean': fold_df['MAE'].mean(),
            'CV_MAE_Std': fold_df['MAE'].std(),
            'CV_RMSE_Mean': fold_df['RMSE'].mean(),
            'CV_RMSE_Std': fold_df['RMSE'].std(),
            'CV_R²_Mean': fold_df['R²'].mean(),
            'CV_R²_Std': fold_df['R²'].std(),
            'CV_MAPE_Mean': fold_df['MAPE'].mean(),
            'CV_MAPE_Std': fold_df['MAPE'].std(),
            'Total_Folds': len(fold_scores)
        })
    
    return pd.DataFrame(cv_results), all_fold_results

# Apply cross-validation to your models
print("Time Series Cross-Validation Evaluation")
print("="*80)

# Create models dictionary
models_dict = {
    'TCN': tcn_model,
    'BiLSTM': bilstm_model,
    'LSTM': lstm_model
}

# Perform cross-validation on training data
cv_results_df, all_fold_results = time_series_cv_evaluation(
    models_dict, 
    X_seq_train, 
    X_add_train, 
    y_train, 
    scalers,
    n_splits=5
)

print("\nCross-Validation Results:")
display(cv_results_df.style.format({
    'CV_MAE_Mean': '{:.4f}',
    'CV_MAE_Std': '{:.4f}',
    'CV_RMSE_Mean': '{:.4f}',
    'CV_RMSE_Std': '{:.4f}',
    'CV_R²_Mean': '{:.4f}',
    'CV_R²_Std': '{:.4f}',
    'CV_MAPE_Mean': '{:.2f}',
    'CV_MAPE_Std': '{:.2f}'
}))

# Visualize cross-validation results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

metrics = ['MAE', 'RMSE', 'R²', 'MAPE']
metric_means = [['CV_MAE_Mean', 'CV_MAE_Std'], 
               ['CV_RMSE_Mean', 'CV_RMSE_Std'],
               ['CV_R²_Mean', 'CV_R²_Std'],
               ['CV_MAPE_Mean', 'CV_MAPE_Std']]

colors = ['blue', 'green', 'orange']
models = cv_results_df['Model'].tolist()

for idx, (metric, (mean_col, std_col)) in enumerate(zip(metrics, metric_means)):
    ax = axes[idx//2, idx%2]
    
    means = cv_results_df[mean_col].values
    stds = cv_results_df[std_col].values
    
    bars = ax.bar(models, means, yerr=stds, capsize=5, color=colors, alpha=0.7, edgecolor='black')
    
    ax.set_title(f'Cross-Validation {metric} Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel(f'{metric} {"(Ah)" if metric in ["MAE", "RMSE"] else "(%)" if metric == "MAPE" else ""}', 
                  fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, mean, std in zip(bars, means, stds):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + std,
                f'{mean:.3f}±{std:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'cross_validation_results.png'), dpi=300, bbox_inches='tight')
plt.show()

#### **8.4 Statistical Analysis of Model Performance**

In [ ]:
# Statistical significance testing between models
print("\nStatistical Significance Testing (Paired t-test):")
print("-" * 50)

from scipy import stats

# Compare models pairwise
model_pairs = [('TCN', 'BiLSTM'), ('TCN', 'LSTM'), ('BiLSTM', 'LSTM')]

for model1, model2 in model_pairs:
    # Get MAE scores for both models across all folds
    model1_mae = [fold['MAE'] for fold in all_fold_results[model1]]
    model2_mae = [fold['MAE'] for fold in all_fold_results[model2]]
    
    # Perform paired t-test
    t_stat, p_value = stats.ttest_rel(model1_mae, model2_mae)
    
    print(f"{model1} vs {model2}:")
    print(f"  Mean MAE - {model1}: {np.mean(model1_mae):.4f}")
    print(f"  Mean MAE - {model2}: {np.mean(model2_mae):.4f}")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_value:.4f}")
    print(f"  Significant difference: {'Yes' if p_value < 0.05 else 'No'}")
    print(f"  Effect size (Cohen's d): {(np.mean(model1_mae) - np.mean(model2_mae)) / np.sqrt((np.std(model1_mae)**2 + np.std(model2_mae)**2) / 2):.4f}")
    print()

# Model stability analysis
print("Model Stability Analysis (Coefficient of Variation):")
print("-" * 50)

for _, row in cv_results_df.iterrows():
    cv_mae = row['CV_MAE_Std'] / row['CV_MAE_Mean'] * 100
    cv_rmse = row['CV_RMSE_Std'] / row['CV_RMSE_Mean'] * 100
    cv_r2 = row['CV_R²_Std'] / abs(row['CV_R²_Mean']) * 100 if row['CV_R²_Mean'] != 0 else np.inf
    
    print(f"{row['Model']}:")
    print(f"  MAE CV: {cv_mae:.2f}%")
    print(f"  RMSE CV: {cv_rmse:.2f}%") 
    print(f"  R² CV: {cv_r2:.2f}%")
    print(f"  Overall Stability: {'High' if cv_mae < 10 else 'Medium' if cv_mae < 20 else 'Low'}")
    print()

# Create fold-wise comparison visualization
print("Fold-wise Performance Visualization:")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, metric in enumerate(['MAE', 'RMSE', 'R²']):
    ax = axes[idx]
    
    for model_name, color in zip(['TCN', 'BiLSTM', 'LSTM'], ['blue', 'green', 'orange']):
        fold_values = [fold[metric] for fold in all_fold_results[model_name]]
        folds = list(range(1, len(fold_values) + 1))
        
        ax.plot(folds, fold_values, 'o-', color=color, label=model_name, linewidth=2, markersize=6)
    
    ax.set_xlabel('Fold Number', fontsize=12, fontweight='bold')
    ax.set_ylabel(f'{metric} {"(Ah)" if metric in ["MAE", "RMSE"] else ""}', fontsize=12, fontweight='bold')
    ax.set_title(f'Fold-wise {metric} Comparison', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(folds)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'fold_wise_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

### **Save Model Results**

In [ ]:
# Create model directory
MODEL_DIR = os.path.join(EXPORT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

def save_model_and_scalers(model, model_name, scalers, history=None):
    """
    Save trained model, scalers, and training history
    """
    # Save model in Keras format
    model_path = os.path.join(MODEL_DIR, f'{model_name}.keras')
    model.save(model_path)
    print(f"Model saved to: {model_path}")
    
    # Save scalers
    scalers_path = os.path.join(MODEL_DIR, f'{model_name}_scalers.pkl')
    with open(scalers_path, 'wb') as f:
        pickle.dump(scalers, f)
    print(f"Scalers saved to: {scalers_path}")
    
    # Save history if provided
    if history is not None:
        history_path = os.path.join(MODEL_DIR, f'{model_name}_history.pkl')
        with open(history_path, 'wb') as f:
            pickle.dump(history.history, f)
        print(f"Training history saved to: {history_path}")
    
    return model_path, scalers_path

print("Saving All Trained Models and Components")
print("=" * 60)

# Save all models
lstm_model_path, lstm_scalers_path = save_model_and_scalers(lstm_model, 'lstm_model', scalers, lstm_history)
tcn_model_path, tcn_scalers_path = save_model_and_scalers(tcn_model, 'tcn_model', scalers, tcn_history)
bilstm_model_path, bilstm_scalers_path = save_model_and_scalers(bilstm_model, 'bilstm_model', scalers, bilstm_history)

# Save training metadata
training_metadata = {
    'train_batteries': train_batteries_final,
    'val_batteries': val_batteries_final,
    'test_batteries': test_batteries,
    'training_samples': len(X_seq_train_split),
    'validation_samples': len(X_seq_val_split),
    'test_samples': len(X_seq_test),
    'sequence_features': ['voltage', 'current', 'temperature'],
    'additional_features': sequence_dataset['feature_names'],
    'lstm_training_time': lstm_training_time,
    'tcn_training_time': tcn_training_time,
    'bilstm_training_time': bilstm_training_time
}

metadata_path = os.path.join(MODEL_DIR, 'training_metadata.pkl')
with open(metadata_path, 'wb') as f:
    pickle.dump(training_metadata, f)
print(f"Training metadata saved to: {metadata_path}")

print(f"\nAll models and components saved to: {MODEL_DIR}")
print("Files saved:")
print("- lstm_model.keras, tcn_model.keras, bilstm_model.keras")
print("- lstm_model_scalers.pkl, tcn_model_scalers.pkl, bilstm_model_scalers.pkl") 
print("- lstm_model_history.pkl, tcn_model_history.pkl, bilstm_model_history.pkl")
print("- training_metadata.pkl")